# Optimizing Multimodal AI: Depth-Aware Fine-Tuning Pipeline
## Green AI Thesis Implementation (T4 Colab) — **v0.8**

**Thesis Scope**: Late-fusion multimodal pipeline fine-tuning LLaVA-1.5-7B with QLoRA on RGB-D data (NYU Depth v2 / SUN RGB-D), evaluated via MADD and efficiency metrics.

**Tracks**: Efficiency (PEFT/Quantization) + Multimodal (RGB-D Fusion)

**Hardware Target**: Single NVIDIA T4 (15 GB VRAM) — Free Colab tier

---
**Work Packages Covered**:
- **WP1**: Environment setup, data loading, toy pipeline
- **WP2**: L1 baseline (LLaVA + LoRA), L1 evaluation, optional L2 (QLoRA 4-bit depth adapters)
- **WP3**: MADD evaluation, efficiency card, W&B logging

---

## Prerequisite (**IMPORTANT**): Kaggle Dataset Access
This notebook requires the NYU Depth v2 dataset hosted on Kaggle:

- Get API Token: Log in to Kaggle, go to [redacted link] > API > Create New Token.
- Download: This will save a kaggle.json file to your computer.
- Upload: You will be prompted to upload this file in Section 4.1 (Data Ingestion).

---

## v0.8 update summary (vs. v0.7)

Version **0.8** mainly fixes the **evaluation methodology** introduced in v0.7, especially for the RGB-only vs. depth-on comparison.

### Main fixes
- **Consistent MADD evaluation pipeline.**  
  The reported headline MADD and the paired significance analysis are now computed from the **same projector-output proxy**, removing the mismatch present in v0.7.

- **Correct RGB-only baseline.**  
  The RGB-only condition is now obtained by forcing the **depth gate** to be effectively closed during the **actual model forward pass**, instead of using an inconsistent proxy path.

- **Aligned per-image metric computation.**  
  The per-image MADD values used for significance testing now apply the **same normalization/post-processing** as the aggregate reported metric.

- **More valid statistical testing.**  
  v0.8 reports paired significance using the corrected per-image differences, including **bootstrap** and **Wilcoxon** testing, so the previous pathological behavior (`p = 1.0` despite large apparent gain) is removed.

- **Improved diagnostic reporting.**  
  The notebook now makes it easier to inspect whether depth is truly contributing, particularly through the learned **gate value** and the corrected RGB-only/depth-on comparison.

### Practical interpretation
These changes do **not** primarily improve the model itself; they improve the **validity of the measurement**. In other words, v0.8 should be interpreted as an **evaluation-fix release** rather than a performance-improvement release.

---
SB (March 26) — v0.8: audit fixes.

## 0. Configuration & Smoke Run Toggle

Set `SMOKE_RUN = True` for a **fast validation pass** (~2-5 min) that tests the entire pipeline end-to-end with minimal data and 1-2 training steps. Set `False` for the full experiment.

In [1]:
# ============================================================
# MASTER CONFIGURATION
# ============================================================
import os

# ──────────────────────────────────────────────────────────────
# SMOKE RUN: Set True for fast pipeline validation (~2-5 min)
#            Set False for full training run
# ──────────────────────────────────────────────────────────────
#SMOKE_RUN = True
SMOKE_RUN = False


# ── Derived Configuration ─────────────────────────────────────
class Config:
    """All hyperparameters in one place. SMOKE_RUN overrides are automatic."""

    # Experiment identity
    project_name = "multimodal-sod"
    run_name = "llava-lora-depth-v1-full"

    # Model
    base_model = "llava-hf/llava-1.5-7b-hf"
    vision_model = "openai/clip-vit-large-patch14-336"
    load_in_4bit = True           # QLoRA (OBJ-3: VRAM reduction)
    bnb_4bit_compute_dtype = "float16"
    bnb_4bit_quant_type = "nf4"

    # LoRA (OBJ-2: parameter-efficient fine-tuning)
    lora_r = 16
    lora_alpha = 32
    lora_dropout = 0.05
    lora_target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "multi_modal_projector.linear_1",          # Cross-modal projection
        "multi_modal_projector.linear_2",
    ]

    # Data
    nyu_max_samples = None if not SMOKE_RUN else 32
    sunrgbd_max_samples = None if not SMOKE_RUN else 32

    # Dataset policy
    primary_rgbd_dataset = "nyuv2"   # "nyuv2" (default) or "nju2k" (debug only)
    toy_rgbd_dataset = "nju2k"       # used ONLY for SMOKE_RUN / stress
    use_toy_for_smoke = True         # if SMOKE_RUN, train on toy dataset (fast)
    run_stress_eval = True           # optionally evaluate on toy dataset (never train) when NYUv2 is primary
    stress_max_samples = 64 if not SMOKE_RUN else 32

    # Dataset roots
    nyuv2_root = "/content/data/NYUv2"
    nju2k_root = "/content/data/NJUD"

    # NYUv2 download (official labeled .mat)
    nyuv2_labeled_url = "https://horatio.cs.nyu.edu/mit/silberman/nyu_depth_v2/nyu_depth_v2_labeled.mat"
    nyuv2_labeled_md5 = "520609c519fba3ba5ac58c8fefcc3530"  # optional integrity check

    # Depth normalization / geometry
    depth_max_m = 10.0              # NYUv2 depths are in meters; normalize by this
    dataloader_num_workers = 0      # safer for h5py/.mat access in Colab
    image_size = 336              # CLIP-ViT-L/14@336px
    depth_size = 336
    #max_length = 256 if not SMOKE_RUN else 768
    #max_length = 768 if not SMOKE_RUN else 256
    max_length = 768

    # Training
    num_epochs = 3 if not SMOKE_RUN else 1
    batch_size = 2                # T4 safe with 4-bit
    gradient_accumulation_steps = 8 if not SMOKE_RUN else 1
    learning_rate = 1e-4
    warmup_ratio = 0.03
    weight_decay = 0.0
    max_steps = -1 if not SMOKE_RUN else 2
    gradient_checkpointing = True
    fp16 = True
    bf16 = False                  # T4 does not support bf16

    # Evaluation
    eval_steps = 50 if not SMOKE_RUN else 1
    save_steps = 50 if not SMOKE_RUN else 999999
    logging_steps = 10 if not SMOKE_RUN else 1

    # Output
    output_dir = "/content/mm-sod-output"
    hub_model_id = None           # Set to push to HF Hub

    # W&B
    #use_wandb = False if SMOKE_RUN else True
    use_wandb = True
    wandb_project = project_name

    # Reproducibility
    seed = 42

    # Provenance tracking (audit fix #1)
    _actual_train_dataset = None  # set at runtime
    _actual_eval_dataset = None   # set at runtime

    @property
    def train_dataset_name(self):
        return self._actual_train_dataset or self.primary_rgbd_dataset

    @property
    def eval_dataset_name(self):
        return self._actual_eval_dataset or self.primary_rgbd_dataset

cfg = Config()

print(f"{'='*60}")
print(f"  MODE: {'🔥 SMOKE RUN (fast validation)' if SMOKE_RUN else '🚀 FULL TRAINING RUN'}")
print(f"  Model: {cfg.base_model}")
print(f"  4-bit: {cfg.load_in_4bit} | LoRA r={cfg.lora_r}")
print(f"  Epochs: {cfg.num_epochs} | Max steps: {cfg.max_steps}")
print(f"  Batch: {cfg.batch_size} x {cfg.gradient_accumulation_steps} accum")
print(f"  NYU samples: {cfg.nyu_max_samples or 'ALL'}")
print(f"  W&B: {cfg.use_wandb}")
print(f"{'='*60}")

  MODE: 🚀 FULL TRAINING RUN
  Model: llava-hf/llava-1.5-7b-hf
  4-bit: True | LoRA r=16
  Epochs: 3 | Max steps: -1
  Batch: 2 x 8 accum
  NYU samples: ALL
  W&B: True


## 1. Environment Setup (WP1)
Install all dependencies. This cell is idempotent.

In [2]:
%%capture install_output
!pip install -q     torch torchvision torchaudio     transformers>=4.36.0     accelerate>=0.25.0     peft>=0.7.0     bitsandbytes>=0.41.0     datasets     pillow     scikit-image     scipy     h5py     wandb     matplotlib     seaborn     tqdm     einops     sentencepiece     protobuf

# Verify GPU
import torch
assert torch.cuda.is_available(), "❌ No GPU detected! Go to Runtime > Change runtime type > T4 GPU"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")
print(f"✅ PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")


In [3]:
# ============================================================
# IMPORTS
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
import json, time, gc, warnings, random, os, sys
from collections import defaultdict
from dataclasses import dataclass
from typing import Optional, Dict, List, Tuple, Any

from transformers import (
    AutoProcessor,
    AutoTokenizer,
    LlavaForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
)
from datasets import load_dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
os.makedirs(cfg.output_dir, exist_ok=True)

# Reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)
print("✅ All imports successful")


✅ All imports successful


## 2. Experiment Tracking (W&B)
Mandatory for all runs following an accountability protocol. Disabled automatically in SMOKE_RUN.

In [4]:
if cfg.use_wandb:
    import wandb
    from google.colab import userdata
    wandb.login(key=userdata.get("WANDB_API_KEY"))
    print("✓ W&B authenticated via Colab secret")
    wandb.init(
        project=cfg.wandb_project,
        name=cfg.run_name + ("_smoke" if SMOKE_RUN else ""),
        config=vars(cfg),
        tags=["smoke"] if SMOKE_RUN else ["full", "L1"],
    )
    # ── P0.1: store stable W&B identifiers ──
    cfg.wandb_run_id   = wandb.run.id
    cfg.wandb_run_name = wandb.run.name
    cfg.wandb_run_url  = wandb.run.get_url()
    cfg.wandb_entity   = wandb.run.entity
    cfg.wandb_project_name = wandb.run.project

    print(f"✅ W&B initialized")
    print(f"   W&B: {cfg.wandb_entity}/{cfg.wandb_project_name}/{cfg.wandb_run_id}")
    print(f"   URL: {cfg.wandb_run_url}")

    # Save provenance artifact
    _wandb_prov = {
        "id": cfg.wandb_run_id,
        "name": cfg.wandb_run_name,
        "url": cfg.wandb_run_url,
        "project": cfg.wandb_project_name,
        "entity": cfg.wandb_entity,
    }
    os.makedirs(cfg.output_dir, exist_ok=True)
    with open(os.path.join(cfg.output_dir, "wandb_run.json"), "w") as _f:
        json.dump(_wandb_prov, _f, indent=2)
else:
    cfg.wandb_run_id = None
    cfg.wandb_run_name = None
    cfg.wandb_run_url = None
    cfg.wandb_entity = None
    cfg.wandb_project_name = None
    os.environ["WANDB_DISABLED"] = "true"
    print("⏭️  W&B disabled")


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: carles-font-alavedra (carles-font-alavedra-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ W&B authenticated via Colab secret


wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.


✅ W&B initialized
   W&B: carles-font-alavedra-/multimodal-sod/1jhaoj1d
   URL: https://wandb.ai/carles-font-alavedra-/multimodal-sod/runs/1jhaoj1d


## 3. Core Metrics (WP3)

### 3.1 MADD — Mean Absolute Difference Dissimilarity
The primary geometric fidelity metric (Ch. 3.1.2 of thesis). More robust to Kinect sensor outliers than MSE.

$$\text{MADD}(\hat{D}, D) = \frac{1}{N} \sum_{i=1}^{N} |\hat{D}_i - D_i|$$


In [5]:
# ============================================================
# evaluation/madd_metric.py — Thesis OBJ-3 metric
# ============================================================

class MADDMetric:
    """
    Mean Absolute Difference Dissimilarity (MADD) for depth evaluation.

    Properties:
      - L1-norm metric: robust to outliers (sensor artifacts in Kinect data)
      - Correlates with human perceptual judgment of geometric correctness
      - Key hypothesis: 4-bit quantization degrades MADD linearly,
        pruning degrades exponentially

    Reference: Thesis Ch. 3.1.2
    """

    def __init__(self):
        self.reset()

    def reset(self):
        self._sum = 0.0
        self._count = 0

    def update(self, pred_depth: torch.Tensor, gt_depth: torch.Tensor,
               valid_mask: Optional[torch.Tensor] = None):
        """
        Args:
            pred_depth: (B, H, W) predicted depth map
            gt_depth:   (B, H, W) ground-truth depth map
            valid_mask: (B, H, W) boolean mask — True for valid pixels
                        (filters Kinect holes / zero-depth artifacts)
        """
        abs_diff = torch.abs(pred_depth.float() - gt_depth.float())
        if valid_mask is not None:
            abs_diff = abs_diff[valid_mask]

        self._sum += abs_diff.sum().item()
        self._count += abs_diff.numel()

    def compute(self) -> float:
        if self._count == 0:
            return 0.0
        return self._sum / self._count

    @staticmethod
    def single(pred_depth: torch.Tensor, gt_depth: torch.Tensor,
               valid_mask: Optional[torch.Tensor] = None) -> float:
        """One-shot MADD for a single batch."""
        abs_diff = torch.abs(pred_depth.float() - gt_depth.float())
        if valid_mask is not None:
            abs_diff = abs_diff[valid_mask]
        return abs_diff.mean().item()


# --- Additional depth metrics for comprehensive evaluation ---

def compute_depth_metrics(pred: torch.Tensor, gt: torch.Tensor,
                          valid_mask: Optional[torch.Tensor] = None) -> Dict[str, float]:
    """
    Full suite of depth evaluation metrics.
    Returns MADD (primary), plus RMSE, delta thresholds for comparison.
    """
    pred_f = pred.float()
    gt_f = gt.float()

    if valid_mask is not None:
        pred_f = pred_f[valid_mask]
        gt_f = gt_f[valid_mask]

    # Avoid division by zero
    gt_f = torch.clamp(gt_f, min=1e-6)
    pred_f = torch.clamp(pred_f, min=1e-6)

    abs_diff = torch.abs(pred_f - gt_f)
    sq_diff = (pred_f - gt_f) ** 2

    # Primary metric
    madd = abs_diff.mean().item()

    # Secondary metrics
    rmse = torch.sqrt(sq_diff.mean()).item()

    # Threshold accuracy (δ < 1.25^k)
    ratio = torch.max(pred_f / gt_f, gt_f / pred_f)
    delta_1 = (ratio < 1.25).float().mean().item()
    delta_2 = (ratio < 1.25 ** 2).float().mean().item()
    delta_3 = (ratio < 1.25 ** 3).float().mean().item()

    # Relative metrics
    abs_rel = (abs_diff / gt_f).mean().item()
    sq_rel = (sq_diff / gt_f).mean().item()

    return {
        "madd": madd,           # PRIMARY (thesis metric)
        "rmse": rmse,
        "abs_rel": abs_rel,
        "sq_rel": sq_rel,
        "delta_1.25": delta_1,
        "delta_1.25^2": delta_2,
        "delta_1.25^3": delta_3,
    }

print("✅ MADD metric + depth evaluation suite defined")


✅ MADD metric + depth evaluation suite defined


### 3.2 Efficiency Tracker
Measures the "Quality vs. Compute" trade-off (OBJ-3). Tracks VRAM, latency, and throughput per run.

In [6]:
# ============================================================
# evaluation/efficiency_card.py — OBJ-3 "Quality vs Compute"
# ============================================================

class EfficiencyTracker:
    """
    Wraps inference/training to capture:
      - Peak VRAM usage (MB)
      - Latency per sample (ms)
      - Throughput (samples/sec)
      - Total time (sec)

    Usage:
        tracker = EfficiencyTracker()
        tracker.start()
        # ... run inference ...
        card = tracker.stop(num_samples=100)
    """

    def __init__(self, device: str = "cuda"):
        self.device = device
        self._start_time = 0.0
        self._cards: List[Dict] = []

    def start(self):
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        self._start_time = time.time()

    def stop(self, num_samples: int, tag: str = "") -> Dict[str, float]:
        torch.cuda.synchronize()
        elapsed = time.time() - self._start_time
        peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 2)  # MB

        card = {
            "tag": tag,
            "num_samples": num_samples,
            "total_time_s": round(elapsed, 3),
            "latency_ms": round((elapsed / max(num_samples, 1)) * 1000, 2),
            "throughput_samples_s": round(num_samples / max(elapsed, 1e-6), 2),
            "peak_vram_mb": round(peak_vram, 1),
        }
        self._cards.append(card)
        return card

    def summary(self) -> List[Dict]:
        return self._cards

    def print_card(self, card: Dict):
        print(f"\n{'─'*50}")
        print(f"  ⚡ EFFICIENCY CARD: {card.get('tag', 'run')}")
        print(f"{'─'*50}")
        print(f"  Samples:     {card['num_samples']}")
        print(f"  Total time:  {card['total_time_s']:.1f} s")
        print(f"  Latency:     {card['latency_ms']:.1f} ms/sample")
        print(f"  Throughput:  {card['throughput_samples_s']:.1f} samples/s")
        print(f"  Peak VRAM:   {card['peak_vram_mb']:.0f} MB")
        print(f"{'─'*50}\n")

tracker = EfficiencyTracker()
print("✅ EfficiencyTracker defined")


✅ EfficiencyTracker defined


In [7]:
# ============================================================
# evaluation/stability_tracker.py — OBJ-1 pipeline stability rate
# ============================================================

class PipelineStabilityTracker:
    """
    Tracks pipeline stability as success_rate = successes / attempts,
    with explicit failure logging per stage (ingest, tokenize, forward).
    OBJ-1 target: ≥ 85% stability on NYUv2 by Gate 1.
    """

    def __init__(self):
        self.stages = defaultdict(lambda: {"attempts": 0, "successes": 0, "failures": []})

    def record_success(self, stage: str):
        self.stages[stage]["attempts"] += 1
        self.stages[stage]["successes"] += 1

    def record_failure(self, stage: str, error: str, sample_id: Any = None):
        self.stages[stage]["attempts"] += 1
        self.stages[stage]["failures"].append({
            "sample_id": sample_id,
            "error": str(error)[:200],
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        })

    def stability_rate(self, stage: str = None) -> float:
        """Returns success_rate for a stage, or overall if stage=None."""
        if stage:
            s = self.stages[stage]
            return s["successes"] / max(s["attempts"], 1)
        total_att = sum(s["attempts"] for s in self.stages.values())
        total_suc = sum(s["successes"] for s in self.stages.values())
        return total_suc / max(total_att, 1)

    def summary(self) -> Dict[str, Any]:
        out = {}
        for stage, s in self.stages.items():
            rate = s["successes"] / max(s["attempts"], 1)
            out[stage] = {
                "attempts": s["attempts"],
                "successes": s["successes"],
                "failures": len(s["failures"]),
                "stability_rate": round(rate, 4),
                "failure_log": s["failures"][-5:],  # last 5 failures
            }
        out["overall_stability_rate"] = round(self.stability_rate(), 4)
        return out

    def meets_obj1(self, threshold: float = 0.85) -> bool:
        return self.stability_rate() >= threshold

    def print_summary(self):
        s = self.summary()
        print(f"\n{'─'*55}")
        print(f"  📊 PIPELINE STABILITY REPORT (OBJ-1 target: ≥85%)")
        print(f"{'─'*55}")
        for stage, info in s.items():
            if stage == "overall_stability_rate":
                continue
            status = "✅" if info["stability_rate"] >= 0.85 else "⚠️"
            print(f"  {status} {stage:20s}  {info['successes']}/{info['attempts']} "
                  f"= {info['stability_rate']*100:.1f}%  ({info['failures']} failures)")
            for fail in info["failure_log"]:
                print(f"      ⛔ [{fail.get('sample_id','?')}] {fail['error'][:80]}")
        overall = s["overall_stability_rate"]
        verdict = "✅ PASS" if overall >= 0.85 else "❌ FAIL"
        print(f"\n  Overall: {overall*100:.1f}%  → {verdict}")
        print(f"{'─'*55}\n")


stability_tracker = PipelineStabilityTracker()
print("✅ PipelineStabilityTracker defined")


✅ PipelineStabilityTracker defined


## 4. Data Ingestion (WP1 / WP2)

### 4.1 NYU Depth v2 Loader (PRIMARY)

### 4.1b NJU2K/NJUD Loader (TOY/STRESS only)
1,449 dense RGB-D pairs captured with Microsoft Kinect. Depth in meters (max ~10m).

Key preprocessing per thesis Ch. 3.3.1:
- Use inpainted/labeled subset (no Kinect holes)
- Normalize depth to [0, 1] range
- Generate visual instruction pairs for LLaVA fine-tuning


In [8]:
# ============================================================
# data/download_nyuv2.py — Download NYU Depth v2 (PRIMARY dataset) via Kaggle
# ============================================================
from pathlib import Path
import os
import shutil
from google.colab import files

NYUV2_ROOT = Path(cfg.nyuv2_root)
NYUV2_ROOT.mkdir(parents=True, exist_ok=True)

MAT_PATH = NYUV2_ROOT / "nyu_depth_v2_labeled.mat"

if not MAT_PATH.exists():
    print(f"⬇️  Downloading NYUv2 labeled dataset via Kaggle...")

    # 1. Setup Kaggle credentials
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    kaggle_json = kaggle_dir / "kaggle.json"

    if not kaggle_json.exists():
        print("Please upload 'kaggle.json' (Kaggle API token) from your computer:")
        uploaded = files.upload()
        for filename in uploaded.keys():
            if "kaggle.json" in filename.lower():
                shutil.move(filename, str(kaggle_json))
                break

        if not kaggle_json.exists():
             raise RuntimeError("kaggle.json not found! Please upload it.")

    # Secure permissions
    kaggle_json.chmod(0o600)

    # 2. Download from Kaggle
    print("   Running kaggle datasets download...")
    !pip install -q kaggle
    !kaggle datasets download -d rmzhang0526/nyu-depth-v2-labeled -p {str(NYUV2_ROOT)}

    # 3. Unzip
    zip_path = NYUV2_ROOT / "nyu-depth-v2-labeled.zip"
    if zip_path.exists():
        print("   Unzipping...")
        !unzip -n {str(zip_path)} -d {str(NYUV2_ROOT)}

    # 4. Verify/Move
    # Ensure the file is at MAT_PATH
    if not MAT_PATH.exists():
        # Look for it recursively
        found = list(NYUV2_ROOT.rglob("nyu_depth_v2_labeled.mat"))
        if found:
            print(f"   Moving {found[0]} to {MAT_PATH}...")
            shutil.move(str(found[0]), str(MAT_PATH))

    if not MAT_PATH.exists():
        raise RuntimeError(f"Download/Extract finished but {MAT_PATH} not found.")

    print(f"✅ Successfully downloaded and extracted: {MAT_PATH}")

else:
    print(f"✅ Found: {MAT_PATH} ({MAT_PATH.stat().st_size/1e9:.2f} GB)")

⬇️  Downloading NYUv2 labeled dataset via Kaggle...
Please upload 'kaggle.json' (Kaggle API token) from your computer:


Saving kaggle.json to kaggle.json
   Running kaggle datasets download...
Dataset URL: https://www.kaggle.com/datasets/rmzhang0526/nyu-depth-v2-labeled
License(s): unknown
100% 2.76G/2.76G [00:35<00:00, 84.1MB/s]

   Unzipping...
Archive:  /content/data/NYUv2/nyu-depth-v2-labeled.zip
  inflating: /content/data/NYUv2/nyu_depth_v2_labeled.mat  
✅ Successfully downloaded and extracted: /content/data/NYUv2/nyu_depth_v2_labeled.mat


In [9]:
# ============================================================
# data/download_nju2k.py — Download NJU2K/NJUD (TOY/STRESS dataset only)
# ============================================================
!pip -q install -U gdown
from pathlib import Path
import zipfile, shutil
import gdown

NJUD_ROOT = Path("/content/data/NJUD")
NJUD_ROOT.mkdir(parents=True, exist_ok=True)
ZIP_PATH = NJUD_ROOT.parent / "NJU2K.zip"

GDRIVE_ID_PRIMARY = "1R1O2dWr6HqpTOiDn6hZxUWTesOSJteQo"

def _has_expected_structure(p: Path) -> bool:
    return (p / "RGB").is_dir() and (p / "depth").is_dir() and (p / "GT").is_dir()

def _normalize_subdir_names(root: Path):
    renames = {
        "Depth": "depth", "DEPTH": "depth",
        "Gt": "GT", "gt": "GT", "GTs": "GT", "Mask": "GT", "mask": "GT",
    }
    for src_name, dst_name in renames.items():
        src = root / src_name
        dst = root / dst_name
        if src.exists() and not dst.exists():
            src.rename(dst)

if not _has_expected_structure(NJUD_ROOT):
    print("Downloading NJUD/NJU2K...")
    url = f"https://drive.google.com/uc?id={GDRIVE_ID_PRIMARY}"
    try:
        gdown.download(url, str(ZIP_PATH), quiet=False, fuzzy=True)
    except Exception as e:
        raise RuntimeError(
            f"gdown failed for primary mirror ({GDRIVE_ID_PRIMARY}).\n"
            "Try: (1) open the URL in a browser and download manually, or\n"
            "     (2) make a copy to your own Drive and use that file's ID.\n"
            f"Original error: {e}"
        )
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(str(NJUD_ROOT.parent))
    ZIP_PATH.unlink(missing_ok=True)

    found = None
    for p in [NJUD_ROOT.parent] + list(NJUD_ROOT.parent.iterdir()):
        if p.is_dir():
            _normalize_subdir_names(p)
            if _has_expected_structure(p):
                found = p
                break
    if found is None:
        for p in NJUD_ROOT.parent.rglob("*"):
            if p.is_dir():
                _normalize_subdir_names(p)
                if _has_expected_structure(p):
                    found = p
                    break
    if found is None:
        raise RuntimeError("Extraction succeeded but could not locate RGB/depth/GT folders.")
    if found.resolve() != NJUD_ROOT.resolve():
        if NJUD_ROOT.exists():
            shutil.rmtree(NJUD_ROOT)
        shutil.move(str(found), str(NJUD_ROOT))

for sub in ["RGB", "depth", "GT"]:
    p = NJUD_ROOT / sub
    n = len(list(p.glob("*"))) if p.exists() else 0
    print(f"{sub}: {n} files {'✓' if n > 0 else '✗ MISSING'}")
print("NJUD_ROOT =", NJUD_ROOT)


Downloading...
From (original): https://drive.google.com/uc?id=1R1O2dWr6HqpTOiDn6hZxUWTesOSJteQo
From (redirected): https://drive.google.com/uc?id=1R1O2dWr6HqpTOiDn6hZxUWTesOSJteQo&confirm=t&uuid=161d42bf-9e7a-402e-b095-d07407842446
To: /content/data/NJU2K.zip
100%|██████████| 1.66G/1.66G [00:20<00:00, 80.5MB/s]


Extracting...
RGB: 3970 files ✓
depth: 1985 files ✓
GT: 2003 files ✓
NJUD_ROOT = /content/data/NJUD


In [10]:
# ============================================================
# data/rgbd_loaders.py — NYUv2 (PRIMARY) + NJU2K (TOY/STRESS)
# ============================================================
import h5py
import numpy as np
from pathlib import Path
from PIL import Image
from skimage.transform import resize as sk_resize
from torch.utils.data import Dataset

# -----------------------------
# NYUv2 (PRIMARY) — official labeled .mat
# -----------------------------
class NYUv2LabeledMatDataset(Dataset):
    '''
    NYU Depth v2 labeled dataset loader from the official nyu_depth_v2_labeled.mat.

    - Reads RGB + inpainted depth (meters) from the .mat (HDF5) file.
    - Normalizes depth to [0,1] for the depth encoder using cfg.depth_max_m.
    - Generates depth-aware instruction Q/A pairs for LLaVA fine-tuning.

    Notes:
    - We avoid `trust_remote_code` by reading the official .mat directly.
    - For stability with h5py, prefer cfg.dataloader_num_workers=0.
    '''

    TEMPLATES = [
        {"q": "Describe the depth structure of this indoor scene.",
         "a": "This indoor scene has depths ranging from {min_d:.2f}m to {max_d:.2f}m, with an average depth of {mean_d:.2f}m. {detail}"},
        {"q": "How far away are the nearest and farthest objects in this image?",
         "a": "The nearest object is approximately {min_d:.2f} meters away, while the farthest point is at {max_d:.2f} meters. The median depth is {median_d:.2f}m."},
        {"q": "What is the approximate distance to the center of this scene?",
         "a": "The center region depth is approximately {center_d:.2f}m. Overall scene depth averages {mean_d:.2f}m with objects between {min_d:.2f}m and {max_d:.2f}m."},
    ]

    def __init__(self, processor, split="train", max_samples=None,
                 image_size=336, depth_size=336, data_root=None):
        self.processor = processor
        self.image_size = image_size
        self.depth_size = depth_size

        root = Path(data_root) if data_root else Path(cfg.nyuv2_root)
        self.mat_path = root / "nyu_depth_v2_labeled.mat"
        assert self.mat_path.exists(), (
            f"NYUv2 .mat not found: {self.mat_path}\n"
            "Run the 'download_nyuv2.py' cell above."
        )

        with h5py.File(self.mat_path, "r") as f:
            assert "images" in f and "depths" in f, f"Unexpected keys: {list(f.keys())[:20]}"
            self._img_shape = tuple(f["images"].shape)
            self._dep_shape = tuple(f["depths"].shape)
            self._n = self._infer_n(self._img_shape)

        # 90/10 deterministic split by index
        split_idx = int(0.9 * self._n)
        if split == "train":
            indices = list(range(0, split_idx))
        elif split in ("val", "validation", "test"):
            indices = list(range(split_idx, self._n))
        else:
            indices = list(range(0, self._n))

        if max_samples is not None:
            indices = indices[:max_samples]
        self.indices = indices

        print(f"📂 NYUv2: mat={self.mat_path.name} | total={self._n} | {split}={len(self.indices)}")
        print(f"   images shape={self._img_shape} | depths shape={self._dep_shape}")

    def _infer_n(self, shape):
        # Common layouts: (N,3,640,480) or (3,640,480,N) or (480,640,3,N)
        for dim in shape:
            if dim == 1449:
                return dim
        return shape[0] if shape[0] > 1000 else shape[-1]

    def __len__(self):
        return len(self.indices)

    def _slice_var(self, var, idx, n):
        # Support: axis0=N or last=N
        if var.ndim >= 1 and var.shape[0] == n:
            return var[idx]
        if var.ndim >= 1 and var.shape[-1] == n:
            return var[..., idx]
        return var[idx]

    def _read_image_depth(self, real_idx: int):
        with h5py.File(self.mat_path, "r") as f:
            img_raw = self._slice_var(f["images"], real_idx, self._n)
            dep_raw = self._slice_var(f["depths"], real_idx, self._n)

        img_raw = np.array(img_raw)
        dep_raw = np.array(dep_raw, dtype=np.float32)

        # Normalize orientation to (480,640,3) / (480,640)
        if img_raw.ndim == 3 and img_raw.shape[0] == 3:
            img = np.rot90(img_raw.transpose(1, 2, 0), -1)
        elif img_raw.ndim == 3 and img_raw.shape[-1] == 3:
            img = np.rot90(img_raw, -1)
        elif img_raw.ndim == 4 and img_raw.shape[0] == 3:
            img = np.rot90(img_raw[:, :, :, 0].transpose(1, 2, 0), -1)
        else:
            raise ValueError(f"Unexpected NYUv2 image slice shape: {img_raw.shape}")

        if dep_raw.ndim == 2:
            depth_m = np.rot90(dep_raw, -1)
        elif dep_raw.ndim == 3:
            depth_m = np.rot90(dep_raw[:, :, 0], -1)
        else:
            raise ValueError(f"Unexpected NYUv2 depth slice shape: {dep_raw.shape}")

        return img.astype(np.uint8), depth_m.astype(np.float32)

    def _get_depth_stats_m(self, depth_m: np.ndarray):
        valid = depth_m[depth_m > 0]
        if valid.size == 0:
            valid = depth_m.reshape(-1)

        h, w = depth_m.shape[:2]
        center_crop = depth_m[h//4:3*h//4, w//4:3*w//4]
        center_valid = center_crop[center_crop > 0]
        center_val = float(np.mean(center_valid)) if center_valid.size > 0 else float(np.mean(valid))

        return {
            "min_d": float(np.percentile(valid, 2)),
            "max_d": float(np.percentile(valid, 98)),
            "mean_d": float(np.mean(valid)),
            "median_d": float(np.median(valid)),
            "std_d": float(np.std(valid)),
            "center_d": center_val,
        }

    def __getitem__(self, idx: int):
        real_idx = self.indices[idx]
        img_u8, depth_m = self._read_image_depth(real_idx)

        image = Image.fromarray(img_u8).convert("RGB").resize(
            (self.image_size, self.image_size), Image.BICUBIC
        )

        depth_m = np.clip(depth_m, 0.0, float(cfg.depth_max_m))
        depth_norm = depth_m / float(cfg.depth_max_m)

        depth_resized = sk_resize(
            depth_norm, (self.depth_size, self.depth_size),
            order=1, preserve_range=True
        ).astype(np.float32)

        stats = self._get_depth_stats_m(depth_m)
        detail_str = ("The depth varies significantly across the scene."
                      if stats["std_d"] > 1.5 else "The depth is relatively uniform across the scene.")
        stats["detail"] = detail_str

        template = self.TEMPLATES[real_idx % len(self.TEMPLATES)]
        question = template["q"]
        answer = template["a"].format(**stats)

        conversation = f"USER: <image>\n{question}\nASSISTANT: {answer}"
        encoding = self.processor(
            text=conversation,
            images=image,
            return_tensors="pt",
            padding="max_length",
            max_length=cfg.max_length,
            truncation=True,
        )

        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        pixel_values = encoding["pixel_values"].squeeze(0)

        labels = input_ids.clone()
        assistant_token = self.processor.tokenizer.encode("ASSISTANT:", add_special_tokens=False)
        for i in range(len(input_ids) - len(assistant_token)):
            if input_ids[i:i+len(assistant_token)].tolist() == assistant_token:
                labels[:i+len(assistant_token)] = -100
                break
        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "pixel_values": pixel_values,
            "labels": labels,
            "depth_map": torch.from_numpy(depth_resized),
        }

# -----------------------------
# NJU2K/NJUD (TOY/STRESS) — relative depth; never claim-bearing
# -----------------------------
class NJU2KStressDataset(Dataset):
    '''
    NJU2K/NJUD RGB-D SOD dataset loader from local files (toy/stress only).
    Depth maps are typically relative / disparity-derived; treat as NON-metric.
    '''

    TEMPLATES = [
        {"q": "Describe the relative depth structure of this scene (normalized depth).",
         "a": "Depth varies from {min_d:.3f} to {max_d:.3f} (normalized). Mean={mean_d:.3f}, median={median_d:.3f}. {detail}"},
        {"q": "Where are the nearest and farthest regions (relative depth)?",
         "a": "Nearest regions are around {min_d:.3f} and farthest around {max_d:.3f} in normalized depth. Std={std_d:.3f}."},
    ]

    IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    def __init__(self, processor, split="train", max_samples=None,
                 image_size=336, depth_size=336, data_root=None):
        self.processor = processor
        self.image_size = image_size
        self.depth_size = depth_size

        root = Path(data_root) if data_root else Path(cfg.nju2k_root)
        rgb_dir = root / "RGB"
        depth_dir = root / "depth"

        assert rgb_dir.is_dir(), f"RGB dir not found: {rgb_dir}"
        assert depth_dir.is_dir(), f"Depth dir not found: {depth_dir}"

        rgb_files = {f.stem: f for f in sorted(rgb_dir.iterdir()) if f.suffix.lower() in self.IMAGE_EXTS}
        depth_files = {f.stem: f for f in sorted(depth_dir.iterdir()) if f.suffix.lower() in self.IMAGE_EXTS}
        common_stems = sorted(set(rgb_files.keys()) & set(depth_files.keys()))
        self.pairs = [(rgb_files[s], depth_files[s]) for s in common_stems]

        n = len(self.pairs)
        split_idx = int(0.9 * n)
        if split == "train":
            self.pairs = self.pairs[:split_idx]
        elif split in ("val", "validation", "test"):
            self.pairs = self.pairs[split_idx:]

        if max_samples is not None:
            self.pairs = self.pairs[:max_samples]

        print(f"📂 NJU2K (stress): root={root} | {split}={len(self.pairs)}")

    def __len__(self):
        return len(self.pairs)

    def _get_depth_stats(self, depth_np):
        valid = depth_np[np.isfinite(depth_np)]
        if valid.size == 0:
            valid = depth_np.reshape(-1)
        return {
            "min_d": float(np.percentile(valid, 2)),
            "max_d": float(np.percentile(valid, 98)),
            "mean_d": float(np.mean(valid)),
            "median_d": float(np.median(valid)),
            "std_d": float(np.std(valid)),
        }

    def __getitem__(self, idx: int):
        rgb_path, depth_path = self.pairs[idx]

        image = Image.open(rgb_path).convert("RGB").resize(
            (self.image_size, self.image_size), Image.BICUBIC
        )

        depth_img = Image.open(depth_path)
        depth_np = np.array(depth_img).astype(np.float32)
        if depth_np.ndim == 3:
            depth_np = depth_np.mean(axis=-1)

        dmin, dmax = float(np.nanmin(depth_np)), float(np.nanmax(depth_np))
        if np.isfinite(dmax - dmin) and (dmax - dmin) > 1e-6:
            depth_np = (depth_np - dmin) / (dmax - dmin)
        else:
            depth_np = np.zeros_like(depth_np, dtype=np.float32)

        depth_resized = sk_resize(
            depth_np, (self.depth_size, self.depth_size),
            order=1, preserve_range=True
        ).astype(np.float32)

        stats = self._get_depth_stats(depth_np)
        detail_str = ("Depth varies strongly across the scene."
                      if stats["std_d"] > 0.25 else "Depth is relatively uniform.")
        stats["detail"] = detail_str

        template = self.TEMPLATES[idx % len(self.TEMPLATES)]
        question = template["q"]
        answer = template["a"].format(**stats)

        conversation = f"USER: <image>\n{question}\nASSISTANT: {answer}"
        encoding = self.processor(
            text=conversation,
            images=image,
            return_tensors="pt",
            padding="max_length",
            max_length=cfg.max_length,
            truncation=True,
        )

        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        pixel_values = encoding["pixel_values"].squeeze(0)

        labels = input_ids.clone()
        assistant_token = self.processor.tokenizer.encode("ASSISTANT:", add_special_tokens=False)
        for i in range(len(input_ids) - len(assistant_token)):
            if input_ids[i:i+len(assistant_token)].tolist() == assistant_token:
                labels[:i+len(assistant_token)] = -100
                break
        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "pixel_values": pixel_values,
            "labels": labels,
            "depth_map": torch.from_numpy(depth_resized),
        }

print("✅ NYUv2LabeledMatDataset + NJU2KStressDataset defined")

✅ NYUv2LabeledMatDataset + NJU2KStressDataset defined


### 4.2 SUN RGB-D Visual Instruction Builder
Converts SUN RGB-D annotations into LLaVA-compatible instruction tuning format (Ch. 4.2.3).

In [11]:
# ============================================================
# data/sun_rgbd_builder.py — SUN RGB-D instruction generation
# ============================================================

class SUNRGBDInstructionDataset(Dataset):
    """
    SUN RGB-D → Visual Instruction pairs.

    Converts scene annotations (2D polygons + 3D bounding boxes)
    into Q/A pairs for depth-aware instruction tuning.

    For T4 constraints, uses the HuggingFace-hosted subset.
    """

    SCENE_TEMPLATES = [
        {
            "q": "What type of indoor scene is this and describe its depth characteristics.",
            "a": "This appears to be a {scene} scene. The depth ranges from "
                 "{min_d:.2f}m to {max_d:.2f}m with an average of {mean_d:.2f}m."
        },
        {
            "q": "Estimate the room dimensions based on the depth information.",
            "a": "Based on the depth map, the visible space extends approximately "
                 "{max_d:.2f}m deep. The depth variation of {std_d:.2f}m suggests "
                 "a {'spacious' if {max_d} > 4 else 'compact'} environment."
        },
        {
            "q": "Identify the nearest and farthest surfaces in this scene.",
            "a": "The nearest surface is at approximately {min_d:.2f}m (likely "
                 "furniture or a wall edge), while the farthest visible surface "
                 "is at {max_d:.2f}m. Mean scene depth: {mean_d:.2f}m."
        },
    ]

    def __init__(self, processor, split="train", max_samples=None,
                 image_size=336, depth_size=336):
        self.processor = processor
        self.image_size = image_size
        self.depth_size = depth_size

        print(f"📂 Loading SUN RGB-D ({split})...")
        try:
            ds = load_dataset("kirill-vish/sun_rgbd_test",
                              split="test" if split == "test" else "train")
        except Exception:
            print("   ⚠️ SUN RGB-D not available. Generating synthetic depth instructions from NYU.")
            ds = load_dataset("sayakpaul/nyu_depth_v2", split=split)

        if max_samples is not None:
            ds = ds.select(range(min(max_samples, len(ds))))

        self.data = ds
        self._scene_labels = [
            "bedroom", "kitchen", "living room", "bathroom", "office",
            "classroom", "corridor", "dining room", "study", "lab"
        ]
        print(f"   ✅ Loaded {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # RGB
        image = item.get("image", item.get("rgb", None))
        if not isinstance(image, Image.Image):
            image = Image.fromarray(np.array(image))
        image = image.convert("RGB").resize(
            (self.image_size, self.image_size), Image.BICUBIC
        )

        # Depth
        depth = item.get("depth_map", item.get("depth", None))
        if isinstance(depth, Image.Image):
            depth_np = np.array(depth).astype(np.float32)
        else:
            depth_np = np.array(depth).astype(np.float32)

        if depth_np.max() > 1.0:
            depth_np = depth_np / max(depth_np.max(), 1e-6)

        from skimage.transform import resize as sk_resize
        depth_resized = sk_resize(depth_np, (self.depth_size, self.depth_size),
                                   order=1, preserve_range=True).astype(np.float32)

        # Stats
        valid = depth_np[depth_np > 0] if (depth_np > 0).any() else depth_np.flatten()
        stats = {
            "min_d": float(np.percentile(valid, 2)),
            "max_d": float(np.percentile(valid, 98)),
            "mean_d": float(np.mean(valid)),
            "std_d": float(np.std(valid)),
            "scene": item.get("scene", self._scene_labels[idx % len(self._scene_labels)]),
        }

        template = self.SCENE_TEMPLATES[idx % len(self.SCENE_TEMPLATES)]
        try:
            question = template["q"]
            answer_tmpl = template["a"]
            if "{'spacious'" in answer_tmpl:
                env = "spacious" if stats["max_d"] > 4 else "compact"
                answer_tmpl = answer_tmpl.split("{")[0] + env + " environment."
                answer = answer_tmpl.format(**stats)
            else:
                answer = template["a"].format(**stats)
        except (KeyError, IndexError):
            question = self.SCENE_TEMPLATES[0]["q"]
            answer = self.SCENE_TEMPLATES[0]["a"].format(**stats)

        conversation = f"USER: <image>\n{question}\nASSISTANT: {answer}"

        encoding = self.processor(
            text=conversation,
            images=image,
            return_tensors="pt",
            padding="max_length",
            max_length=cfg.max_length,
            truncation=True,
        )

        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        pixel_values = encoding["pixel_values"].squeeze(0)

        labels = input_ids.clone()
        assistant_token = self.processor.tokenizer.encode("ASSISTANT:", add_special_tokens=False)
        for i in range(len(input_ids) - len(assistant_token)):
            if input_ids[i:i+len(assistant_token)].tolist() == assistant_token:
                labels[:i+len(assistant_token)] = -100
                break
        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "pixel_values": pixel_values,
            "labels": labels,
            "depth_map": torch.from_numpy(depth_resized),
        }

print("✅ SUNRGBDInstructionDataset defined")


✅ SUNRGBDInstructionDataset defined


## 5. Model Architecture (WP2)

### 5.1 Depth Encoder & Cross-Modal Connector
Implements the **Late Fusion** strategy (thesis Table 3.1, recommended approach):
1. Frozen CLIP-ViT-L/14 → RGB embedding
2. Lightweight Depth Encoder → Depth embedding  
3. Learned projection layer → fused into LLM token space


In [12]:
# ============================================================
# models/depth_encoder.py — Lightweight depth backbone
# models/connectors.py  — Cross-modal projection (RGB+D → LLM)
# ============================================================

class DepthEncoder(nn.Module):
    """
    Lightweight depth encoder for the Late Fusion pipeline.

    Architecture: Compact ConvNet that maps single-channel depth maps
    to an embedding compatible with CLIP's visual feature space.

    Design choice: We use a simple ConvNet instead of a full ViT
    for the depth path because:
      1. T4 VRAM constraint (~15 GB)
      2. Depth maps are geometrically simpler than RGB
      3. The LLM + CLIP backbone already consume most VRAM

    Output: (B, num_patches, embed_dim) matching CLIP's output format.
    """

    def __init__(self, embed_dim: int = 1024, num_patches: int = 576,
                 input_size: int = 336):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_patches = num_patches

        # Efficient ConvNet: depth (1ch) → features
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 7, stride=2, padding=3),   # 336→168
            nn.GELU(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # 168→84
            nn.GELU(),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), # 84→42
            nn.GELU(),
            nn.BatchNorm2d(128),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), # 42→21
            nn.GELU(),
            nn.AdaptiveAvgPool2d((24, 24)),              # → 24×24 = 576 patches
        )

        # Project to CLIP embedding dimension
        self.proj = nn.Linear(256, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, depth: torch.Tensor) -> torch.Tensor:
        if depth.dim() == 3:
            depth = depth.unsqueeze(1)

        input_dtype = depth.dtype
        # Run conv+BN stack in float32 to avoid BN dtype mismatch
        x = self.encoder(depth.float())              # (B, 256, 24, 24)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)             # (B, 576, 256)
        x = x.to(input_dtype)                        # back to original dtype
        x = self.proj(x)                             # (B, 576, embed_dim)
        x = self.norm(x)
        return x
    # def forward(self, depth: torch.Tensor) -> torch.Tensor:
    #     """
    #     Args:
    #         depth: (B, H, W) or (B, 1, H, W) depth map, normalized [0,1]
    #     Returns:
    #         (B, num_patches, embed_dim) depth features
    #     """
    #     if depth.dim() == 3:
    #         depth = depth.unsqueeze(1)  # (B, 1, H, W)

    #     x = self.encoder(depth)                    # (B, 256, 24, 24)
    #     B, C, H, W = x.shape
    #     x = x.flatten(2).transpose(1, 2)           # (B, 576, 256)
    #     x = self.proj(x)                           # (B, 576, embed_dim)
    #     x = self.norm(x)
    #     return x


class DepthFusionConnector(nn.Module):
    """
    Cross-Modal Connector: fuses RGB (from CLIP) and Depth embeddings
    before injection into the LLM.

    Strategy: Gated addition (learnable weighting of depth contribution).
    This allows the model to learn how much geometric info to incorporate
    without disrupting the pre-trained RGB representations.
    """

    def __init__(self, embed_dim: int = 1024):
        super().__init__()
        # Learnable gate: controls depth contribution
        self.gate = nn.Parameter(torch.tensor([-2.0]))  # sigmoid(-10)≈0 → starts RGB-only
        self.depth_proj = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
        )
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, rgb_features: torch.Tensor,
                depth_features: torch.Tensor) -> torch.Tensor:
        """
        Args:
            rgb_features:   (B, N, D) from CLIP vision encoder
            depth_features: (B, N, D) from DepthEncoder
        Returns:
            (B, N, D) fused features
        """
        alpha = torch.sigmoid(self.gate)  # Smooth gate [0, 1]
        depth_proj = self.depth_proj(depth_features)
        fused = (1 - alpha) * rgb_features + alpha * depth_proj
        return self.norm(fused)


# Quick test
_de = DepthEncoder(embed_dim=1024)
_fc = DepthFusionConnector(embed_dim=1024)
_test_depth = torch.randn(2, 1, 336, 336)
_test_rgb = torch.randn(2, 576, 1024)
_d_out = _de(_test_depth)
_fused = _fc(_test_rgb, _d_out)
print(f"✅ DepthEncoder: depth (2,1,336,336) → features {_d_out.shape}")
print(f"✅ DepthFusionConnector: fused output {_fused.shape}")
print(f"   Depth encoder params: {sum(p.numel() for p in _de.parameters()):,}")
del _de, _fc, _test_depth, _test_rgb, _d_out, _fused


✅ DepthEncoder: depth (2,1,336,336) → features torch.Size([2, 576, 1024])
✅ DepthFusionConnector: fused output torch.Size([2, 576, 1024])
   Depth encoder params: 654,784


### 5.2 Model Loading — LLaVA-1.5-7B with QLoRA
Loads the base VLM in 4-bit quantization (NF4) with LoRA adapters on attention + projection layers.

**VRAM Budget** (T4, 15 GB):
- 4-bit LLaVA-7B: ~4-5 GB
- LoRA adapters: ~50 MB  
- Depth encoder: ~5 MB
- Activations + optimizer: ~5-8 GB
- **Total**: ~10-14 GB ✅


In [13]:
# ============================================================
# Load LLaVA-1.5-7B with 4-bit quantization + LoRA
# ============================================================

print("📦 Loading model (this may take 2-5 minutes on first run)...")
print(f"   Base: {cfg.base_model}")
print(f"   4-bit: {cfg.load_in_4bit} | LoRA r={cfg.lora_r}")

# --- Quantization config (QLoRA) ---
bnb_config = None
if cfg.load_in_4bit:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
        bnb_4bit_use_double_quant=True,  # Nested quantization for extra savings
    )

# --- Load processor ---
processor = AutoProcessor.from_pretrained(cfg.base_model)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# --- Load model ---
model = LlavaForConditionalGeneration.from_pretrained(
    cfg.base_model,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

# --- Prepare for k-bit training ---
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=cfg.gradient_checkpointing,
)

# ── P0.2.2: DepthAwareProjector — wraps multi_modal_projector ──
clip_hidden_size = model.config.vision_config.hidden_size
print(f"   CLIP hidden size: {clip_hidden_size}")


class DepthAwareProjector(nn.Module):
    """
    Drop-in replacement for LLaVA's multi_modal_projector that
    fuses RGB patch embeddings with depth patch embeddings BEFORE
    projecting into the LLM space.

    Preserves linear_1 / linear_2 names so LoRA targeting is unchanged.
    """

    def __init__(self, orig_proj, embed_dim: int = 1024):
        super().__init__()
        # Keep original projector submodules by the SAME names
        self.linear_1 = orig_proj.linear_1
        self.linear_2 = orig_proj.linear_2
        self.act = getattr(orig_proj, "act", nn.GELU())

        # New trainable depth modules
        self.depth_encoder = DepthEncoder(embed_dim=embed_dim)
        self.depth_connector = DepthFusionConnector(embed_dim=embed_dim)

        # Cache slot for depth maps (Trainer-compatible injection)
        self._cached_depth_maps = None

    def forward(self, image_features: torch.Tensor) -> torch.Tensor:
        """
        image_features: (B, N, D) from CLIP vision encoder
                        N may be 577 (with CLS) or 576 (patches only)
        """
        x = image_features  # RGB features from CLIP

        if self._cached_depth_maps is not None:
            depth_maps = self._cached_depth_maps
            self._cached_depth_maps = None  # consume & clear

            depth_feats = self.depth_encoder(
                depth_maps.to(dtype=x.dtype, device=x.device)
            )  # (B, 576, D)

            has_cls = (x.shape[1] == depth_feats.shape[1] + 1)
            if has_cls:
                cls_token = x[:, :1, :]
                rgb_patches = x[:, 1:, :]
                fused_patches = self.depth_connector(rgb_patches, depth_feats)
                x = torch.cat([cls_token, fused_patches], dim=1)
            else:
                # Align lengths if needed
                min_len = min(x.shape[1], depth_feats.shape[1])
                x_trim = x[:, :min_len, :]
                d_trim = depth_feats[:, :min_len, :]
                fused = self.depth_connector(x_trim, d_trim)
                if x.shape[1] > min_len:
                    x = torch.cat([fused, x[:, min_len:, :]], dim=1)
                else:
                    x = fused

        # Original projector forward: linear_1 → act → linear_2
        x = self.linear_1(x)
        x = self.act(x)
        x = self.linear_2(x)
        return x


# Replace projector BEFORE LoRA is applied
orig_proj = model.model.multi_modal_projector
depth_aware_proj = DepthAwareProjector(orig_proj, embed_dim=clip_hidden_size)
depth_aware_proj = depth_aware_proj.to(dtype=torch.float16, device=model.device)
model.model.multi_modal_projector = depth_aware_proj.float()
print("✅ DepthAwareProjector installed (preserves linear_1/linear_2 for LoRA)")

# --- Apply LoRA ---
lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=cfg.lora_target_modules,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

# ── P0.2.3: Patch model forward to inject depth_maps into projector cache ──

def _get_projector(m):
    """Resolve the actual DepthAwareProjector through PEFT wrapping."""
    core = m
    for attr in ["base_model", "model"]:
        core = getattr(core, attr, core)
    if hasattr(core, "multi_modal_projector"):
        return core.multi_modal_projector
    # Deeper search
    for name, mod in core.named_modules():
        if isinstance(mod, DepthAwareProjector):
            return mod
    return None

_orig_forward = model.forward

def _depth_injecting_forward(*args, **kwargs):
    depth_maps = kwargs.pop("depth_maps", None)
    proj = _get_projector(model)
    if proj is not None and depth_maps is not None:
        proj._cached_depth_maps = depth_maps
    out = _orig_forward(*args, **kwargs)
    # Clear cache to avoid holding memory
    if proj is not None:
        proj._cached_depth_maps = None
    return out

model.forward = _depth_injecting_forward
print("✅ Forward wrapper installed (depth_maps → projector cache)")

# ── P0.3: Trainable parameter audit including depth modules ──
trainable, total = 0, 0
lora_params, depth_params_count = 0, 0
for n, p in model.named_parameters():
    total += p.numel()
    if p.requires_grad:
        trainable += p.numel()
        if "depth_encoder" in n or "depth_connector" in n:
            depth_params_count += p.numel()
        else:
            lora_params += p.numel()

# Also count depth modules that may be outside PEFT wrapping
proj = _get_projector(model)
if proj is not None:
    for n, p in proj.named_parameters():
        if not p.requires_grad and ("depth_encoder" in n or "depth_connector" in n):
            p.requires_grad_(True)
            depth_params_count += p.numel()
            trainable += p.numel()

print(f"\n{'='*60}")
print(f"  ✅ Model loaded successfully")
print(f"  Total params:         {total:>14,}")
print(f"  Trainable (LoRA):     {lora_params:>14,}")
print(f"  Trainable (depth):    {depth_params_count:>14,}")
print(f"  Trainable (total):    {trainable:>14,} ({100*trainable/total:.2f}%)")
print(f"  VRAM used:            {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"  VRAM reserved:        {torch.cuda.memory_reserved()/1e9:.2f} GB")
print(f"{'='*60}")
model.print_trainable_parameters()


📦 Loading model (this may take 2-5 minutes on first run)...
   Base: llava-hf/llava-1.5-7b-hf
   4-bit: True | LoRA r=16


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

   CLIP hidden size: 1024
✅ DepthAwareProjector installed (preserves linear_1/linear_2 for LoRA)
✅ Forward wrapper installed (depth_maps → projector cache)

  ✅ Model loaded successfully
  Total params:          3,686,049,217
  Trainable (LoRA):         19,349,504
  Trainable (depth):         2,756,033
  Trainable (total):        22,105,537 (0.60%)
  VRAM used:            4.68 GB
  VRAM reserved:        13.29 GB
trainable params: 22,105,537 || all params: 7,085,532,609 || trainable%: 0.3120


In [14]:
# ============================================================
# Depth modules are now inside DepthAwareProjector (Cell 22)
# Expose references for backward compatibility (save, eval)
# ============================================================

proj = _get_projector(model)
depth_encoder = proj.depth_encoder
depth_connector = proj.depth_connector

depth_params = sum(p.numel() for p in depth_encoder.parameters())
conn_params = sum(p.numel() for p in depth_connector.parameters())
print(f"✅ Depth encoder params (inside projector):  {depth_params:,}")
print(f"✅ Fusion connector params:                  {conn_params:,}")
print(f"   Additional VRAM: ~{(depth_params + conn_params) * 2 / 1e6:.1f} MB (FP16)")


✅ Depth encoder params (inside projector):  654,784
✅ Fusion connector params:                  2,101,249
   Additional VRAM: ~5.5 MB (FP16)


## 6. Dataset Preparation

In [15]:
# ============================================================
# Load and prepare datasets (NYUv2 primary; NJU2K toy/stress)
# ============================================================
print(f"\n{'='*60}")
print(f"  Loading datasets (SMOKE_RUN={SMOKE_RUN})")
print(f"{'='*60}\n")

def _make_dataset(name: str, split: str, max_samples):
    name = name.lower()
    if name in ("nyuv2", "nyu", "nyu_depth_v2"):
        return NYUv2LabeledMatDataset(
            processor=processor,
            split=split,
            max_samples=max_samples,
            image_size=cfg.image_size,
            depth_size=cfg.depth_size,
            data_root=cfg.nyuv2_root,
        )
    if name in ("nju2k", "njud"):
        return NJU2KStressDataset(
            processor=processor,
            split=split,
            max_samples=max_samples,
            image_size=cfg.image_size,
            depth_size=cfg.depth_size,
            data_root=cfg.nju2k_root,
        )
    raise ValueError(f"Unknown dataset: {name}")

# --- Primary training dataset selection ---
if SMOKE_RUN and cfg.use_toy_for_smoke:
    primary_name = cfg.toy_rgbd_dataset
    print("⚙️  SMOKE_RUN: using TOY dataset for fast validation:", primary_name)
    cfg._actual_train_dataset = primary_name   # FIX 1: provenance
    cfg._actual_eval_dataset = primary_name
    train_dataset = _make_dataset(primary_name, split="train", max_samples=cfg.nyu_max_samples)
    val_dataset   = _make_dataset(primary_name, split="val",   max_samples=max(4, int((cfg.nyu_max_samples or 32) * 0.2)))
else:
    primary_name = cfg.primary_rgbd_dataset
    print("✅ Using PRIMARY dataset for training:", primary_name)
    cfg._actual_train_dataset = primary_name   # FIX 1: provenance
    cfg._actual_eval_dataset = primary_name
    train_dataset = _make_dataset(primary_name, split="train", max_samples=cfg.nyu_max_samples)
    val_dataset   = _make_dataset(primary_name, split="val",   max_samples=None if not SMOKE_RUN else 4)

# FIX 1: Explicit provenance banner
print(f"\n🏷️  PROVENANCE: training on '{cfg.train_dataset_name}', evaluating on '{cfg.eval_dataset_name}'")
if cfg.train_dataset_name != "nyuv2":
    print("   ⚠️  NOTE: OBJ-1 requires NYUv2. Current run uses toy data and is NOT claim-bearing for OBJ-1.")

# For backwards compatibility with the rest of the notebook
train_subset = train_dataset

print(f"\n📊 Dataset sizes:")
print(f"   Train: {len(train_dataset)} samples")
print(f"   Val:   {len(val_dataset)} samples")

stability_tracker = PipelineStabilityTracker()

# FIX 4 + P0.4: Stability-instrumented data loading test (ingest / tokenize / forward)
print(f"\n🔬 Running stability probe on train dataset...")
n_probe = min(len(train_dataset), 20)
_probe_samples = []
for probe_i in range(n_probe):
    # ── Stage 1: ingest ──
    try:
        sample = train_dataset[probe_i]
        stability_tracker.record_success("ingest")
    except Exception as e:
        stability_tracker.record_failure("ingest", str(e), sample_id=probe_i)
        continue

    # ── Stage 2: tokenize (validate shapes) ──
    try:
        assert "input_ids" in sample, "missing input_ids"
        assert "attention_mask" in sample, "missing attention_mask"
        assert "labels" in sample, "missing labels"
        assert "depth_map" in sample, "missing depth_map"
        assert sample["input_ids"].dim() == 1, f"input_ids rank={sample['input_ids'].dim()}"
        assert sample["attention_mask"].shape == sample["input_ids"].shape, "mask/ids shape mismatch"
        stability_tracker.record_success("tokenize")
        _probe_samples.append(sample)
    except Exception as e:
        stability_tracker.record_failure("tokenize", str(e), sample_id=probe_i)

# ── Stage 3: forward probe (2-4 samples) ──
if len(_probe_samples) >= 2:
    _fwd_batch_samples = _probe_samples[:min(4, len(_probe_samples))]
    try:
        _fwd_batch = collator(_fwd_batch_samples)
        _fwd_batch = {k: v.to("cuda") if hasattr(v, "to") else v for k, v in _fwd_batch.items()}
        model.train()
        with torch.amp.autocast("cuda", dtype=torch.float16):
            _fwd_out = model(**_fwd_batch)
        assert hasattr(_fwd_out, "loss") and _fwd_out.loss is not None, "no loss returned"
        stability_tracker.record_success("forward")
        print(f"   forward probe loss: {_fwd_out.loss.item():.4f}")

        # ── Optional Stage 4: backward probe ──
        try:
            _fwd_out.loss.backward()
            stability_tracker.record_success("backward")
            model.zero_grad()
        except Exception as e:
            stability_tracker.record_failure("backward", str(e))
    except Exception as e:
        stability_tracker.record_failure("forward", str(e))
else:
    stability_tracker.record_failure("forward", "not enough valid samples for forward probe")

del _probe_samples
stability_tracker.print_summary()

# Optional: stress dataset (never used for training)
stress_dataset = None
if (not SMOKE_RUN) and cfg.run_stress_eval:
    try:
        stress_dataset = _make_dataset(cfg.toy_rgbd_dataset, split="val", max_samples=cfg.stress_max_samples)
        print(f"🧪 Stress-eval dataset enabled: {cfg.toy_rgbd_dataset} (val) n={len(stress_dataset)}")
    except Exception as e:
        print("⚠️ Stress dataset not available (download it if you want stress-eval).")
        print("   Error:", e)

# Quick sanity check
sample = train_dataset[0]
print(f"\n🔍 Sample shapes:")
print(f"   input_ids:    {sample['input_ids'].shape}")
print(f"   pixel_values: {sample['pixel_values'].shape}")
print(f"   depth_map:    {sample['depth_map'].shape}")



  Loading datasets (SMOKE_RUN=False)

✅ Using PRIMARY dataset for training: nyuv2
📂 NYUv2: mat=nyu_depth_v2_labeled.mat | total=1449 | train=1304
   images shape=(1449, 3, 640, 480) | depths shape=(1449, 640, 480)
📂 NYUv2: mat=nyu_depth_v2_labeled.mat | total=1449 | val=145
   images shape=(1449, 3, 640, 480) | depths shape=(1449, 640, 480)

🏷️  PROVENANCE: training on 'nyuv2', evaluating on 'nyuv2'

📊 Dataset sizes:
   Train: 1304 samples
   Val:   145 samples

🔬 Running stability probe on train dataset...

───────────────────────────────────────────────────────
  📊 PIPELINE STABILITY REPORT (OBJ-1 target: ≥85%)
───────────────────────────────────────────────────────
  ✅ ingest                20/20 = 100.0%  (0 failures)
  ✅ tokenize              20/20 = 100.0%  (0 failures)
  ⚠️ forward               0/1 = 0.0%  (1 failures)
      ⛔ [None] name 'collator' is not defined

  Overall: 97.6%  → ✅ PASS
───────────────────────────────────────────────────────

📂 NJU2K (stress): root=/conte

In [16]:
# ============================================================
# Custom data collator that handles depth maps
# ============================================================

class MultimodalDepthCollator:
    """Collates batches including depth maps for the depth-aware pipeline."""

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        batch = {
            "input_ids": torch.stack([f["input_ids"] for f in features]),
            "attention_mask": torch.stack([f["attention_mask"] for f in features]),
            "pixel_values": torch.stack([f["pixel_values"] for f in features]),
            "labels": torch.stack([f["labels"] for f in features]),
        }
        # Depth maps kept separate (used by depth encoder, not by HF Trainer directly)
        if "depth_map" in features[0]:
            batch["depth_maps"] = torch.stack([f["depth_map"] for f in features])
        return batch

collator = MultimodalDepthCollator()
print("✅ MultimodalDepthCollator defined")


✅ MultimodalDepthCollator defined


## 7. Training Loop (WP2 — L1 Development)

Fine-tunes LLaVA-1.5-7B with LoRA on depth-aware visual instructions.

**Training Strategy**:
- QLoRA 4-bit base + LoRA r=16 on attention + projection layers
- Gradient checkpointing enabled (saves ~40% VRAM)
- FP16 mixed precision (T4 optimized)
- Gradient accumulation = 8 (effective batch = 16)


In [ ]:
# ============================================================
# Training Arguments & Trainer
# ============================================================

training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    fp16=cfg.fp16,
    bf16=cfg.bf16,
    max_steps=cfg.max_steps,
    logging_steps=cfg.logging_steps,
    eval_strategy="steps" if not SMOKE_RUN else "no",
    eval_steps=cfg.eval_steps if not SMOKE_RUN else None,
    save_strategy="steps" if not SMOKE_RUN else "no",
    save_steps=cfg.save_steps,
    save_total_limit=2,
    gradient_checkpointing=cfg.gradient_checkpointing,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_pin_memory=True,
    dataloader_num_workers=cfg.dataloader_num_workers,
    remove_unused_columns=False,  # Keep depth_maps column
    report_to="wandb" if cfg.use_wandb else "none",
    run_name=cfg.run_name,
    seed=cfg.seed,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",  # Memory-efficient optimizer for QLoRA
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=val_dataset if not SMOKE_RUN else None,
    data_collator=collator,
)

print(f"\n{'='*60}")
print(f"  🚀 Starting {'SMOKE' if SMOKE_RUN else 'FULL'} training")
print(f"  Effective batch size: {cfg.batch_size * cfg.gradient_accumulation_steps}")
print(f"  Max steps: {cfg.max_steps if cfg.max_steps > 0 else 'all'}")
print(f"  Optimizer: paged_adamw_8bit")
print(f"  Scheduler: cosine")
print(f"{'='*60}\n")

# ── P0.2.4: One-batch forward sanity test (depth-on vs depth-off) ──
print("\n🔬 Running depth fusion sanity test...")
_proj = _get_projector(model)
_hook_call_count = {"total": 0, "with_depth": 0}

def _projector_hook(module, input, output):
    _hook_call_count["total"] += 1

_hook_handle = _proj.register_forward_hook(_projector_hook)

model.train()
_sanity_samples = [train_subset[i] for i in range(min(2, len(train_subset)))]
_sanity_batch = collator(_sanity_samples)
_sanity_batch = {k: v.to("cuda") if hasattr(v, "to") else v for k, v in _sanity_batch.items()}

with torch.no_grad():
    # Run 1: with depth
    _hook_call_count["total"] = 0
    out_depth_on = model(**_sanity_batch)
    logits_depth_on = out_depth_on.logits.clone()
    calls_with_depth = _hook_call_count["total"]

    # Run 2: depth disabled (force gate → RGB-only)
    _orig_gate = _proj.depth_connector.gate.data.clone()
    _proj.depth_connector.gate.data.fill_(-100.0)  # sigmoid(-100)≈0
    _hook_call_count["total"] = 0
    out_depth_off = model(**_sanity_batch)
    logits_depth_off = out_depth_off.logits.clone()
    calls_without_depth = _hook_call_count["total"]
    _proj.depth_connector.gate.data.copy_(_orig_gate)

_hook_handle.remove()

_logit_diff = (logits_depth_on - logits_depth_off).abs().max().item()
_alpha_val = torch.sigmoid(_proj.depth_connector.gate).item()
print(f"   Projector hook fired: {calls_with_depth}x (depth-on), {calls_without_depth}x (depth-off)")
print(f"   Current gate alpha: {_alpha_val:.6f}")
print(f"   Max logit difference (depth-on vs depth-off): {_logit_diff:.6f}")
if _alpha_val > 1e-4:
    assert _logit_diff > 0, "FATAL: depth fusion has no effect despite alpha>0"
    print("   ✅ Depth fusion is ACTIVE and changes model output")
else:
    print("   ✅ Gate near zero (expected at init); logits ~identical (correct)")

del _sanity_samples, _sanity_batch, logits_depth_on, logits_depth_off
gc.collect(); torch.cuda.empty_cache()

# Register persistent hook for training monitoring
_train_hook_stats = {"calls": 0, "depth_present": 0}

def _train_projector_hook(module, input, output):
    _train_hook_stats["calls"] += 1
    if hasattr(module, "_cached_depth_maps") and module._cached_depth_maps is not None:
        _train_hook_stats["depth_present"] += 1

_train_hook_handle = _proj.register_forward_hook(_train_projector_hook)

# --- Train ---
tracker.start()
train_result = trainer.train()
train_card = tracker.stop(
    num_samples=len(train_subset) * cfg.num_epochs if cfg.max_steps <= 0
                else cfg.max_steps * cfg.batch_size,
    tag="training"
)
tracker.print_card(train_card)

# --- Log metrics ---
metrics = train_result.metrics
print(f"\n📊 Training metrics:")
for k, v in metrics.items():
    print(f"   {k}: {v:.4f}" if isinstance(v, float) else f"   {k}: {v}")

# ── P0.2.4 / P0.3: Post-training hook & gate audit ──
_train_hook_handle.remove()
print(f"\n   Projector hook stats: {_train_hook_stats['calls']} calls, "
      f"{_train_hook_stats['depth_present']} with depth cached")
_final_alpha = torch.sigmoid(_proj.depth_connector.gate).item()
print(f"   Final gate alpha: {_final_alpha:.6f}")

if cfg.use_wandb:
    import wandb
    wandb.log({
        "efficiency/training": train_card,
        "depth/gate_alpha_final": _final_alpha,
        "depth/projector_calls": _train_hook_stats["calls"],
        "depth/depth_present_calls": _train_hook_stats["depth_present"],
    })


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



  🚀 Starting FULL training
  Effective batch size: 16
  Max steps: all
  Optimizer: paged_adamw_8bit
  Scheduler: cosine


🔬 Running depth fusion sanity test...
   Projector hook fired: 1x (depth-on), 1x (depth-off)
   Current gate alpha: 0.119203
   Max logit difference (depth-on vs depth-off): 1.288098
   ✅ Depth fusion is ACTIVE and changes model output


Step,Training Loss,Validation Loss
50,3.855007,3.782268
100,3.518475,3.513498


In [ ]:
# ============================================================
# Save LoRA adapters (lightweight checkpoint)
# ============================================================
save_path = os.path.join(cfg.output_dir, "lora_adapters")
model.save_pretrained(save_path)
processor.save_pretrained(save_path)

# Save depth encoder separately
torch.save({
    "depth_encoder": depth_encoder.state_dict(),
    "depth_connector": depth_connector.state_dict(),
}, os.path.join(cfg.output_dir, "depth_modules.pt"))

adapter_size = sum(
    os.path.getsize(os.path.join(save_path, f))
    for f in os.listdir(save_path) if os.path.isfile(os.path.join(save_path, f))
) / 1e6

print(f"\n✅ Saved LoRA adapters to: {save_path}")
print(f"   Adapter size: {adapter_size:.1f} MB")
print(f"   Depth modules: {os.path.join(cfg.output_dir, 'depth_modules.pt')}")


## 8. Evaluation (WP3)

### 8.1 MADD Evaluation & Efficiency Card
Systematic evaluation of geometric fidelity and computational cost.

**Metrics**:
- **Primary**: MADD (Mean Absolute Difference Dissimilarity)  
- **Secondary**: RMSE, δ thresholds (1.25, 1.25², 1.25³), Abs Rel
- **Efficiency**: Latency (ms/sample), Throughput (samples/s), Peak VRAM (MB)


In [ ]:
# ============================================================
# Evaluation: MADD + full depth metrics + efficiency
# ============================================================
print(f"\n{'='*60}")
print(f"  📐 Running MADD Evaluation")
print(f"{'='*60}\n")

import math

model.eval()
madd_metric = MADDMetric()
all_depth_metrics = []
depthon_per_sample_madd = []
eval_loader = DataLoader(val_dataset, batch_size=cfg.batch_size,
                         collate_fn=collator, shuffle=False)

tracker.start()

_proj = _get_projector(model)
assert _proj is not None, "DepthAwareProjector not found; cannot run aligned evaluation."


def _normalize_pred_to_gt_range(pred_depth: torch.Tensor,
                                depth_maps: torch.Tensor,
                                valid_mask: torch.Tensor) -> torch.Tensor:
    """Match proxy depth range to GT range per sample using valid GT pixels only."""
    pred_depth = pred_depth.clone()
    for b in range(pred_depth.size(0)):
        vm = valid_mask[b]
        if vm.sum() == 0:
            continue
        gt_valid = depth_maps[b][vm]
        gt_min = gt_valid.min()
        gt_max = gt_valid.max()
        p = pred_depth[b]
        p_min = p.min()
        p_max = p.max()
        if (p_max - p_min) > 1e-8 and (gt_max - gt_min) > 1e-8:
            pred_depth[b] = (p - p_min) / (p_max - p_min + 1e-8) * (gt_max - gt_min) + gt_min
        else:
            pred_depth[b] = torch.full_like(p, gt_min)
    return pred_depth


def _tokens_to_proxy_depth(projector_tokens: torch.Tensor,
                           depth_maps: torch.Tensor) -> torch.Tensor:
    """
    Convert projector output tokens into a spatial proxy depth map.
    Uses the SAME construction for headline MADD and paired significance.
    """
    toks = projector_tokens.detach().float()
    if toks.ndim != 3:
        raise ValueError(f"Unexpected projector token rank: {toks.shape}")

    # Drop CLS token if present (577 -> 576 patches)
    if toks.shape[1] == 577:
        toks = toks[:, 1:, :]

    n_tokens = toks.shape[1]
    side = int(math.sqrt(n_tokens))
    if side * side != n_tokens:
        square_n = side * side
        if square_n <= 0:
            raise ValueError(f"Cannot reshape projector tokens with shape {toks.shape}")
        toks = toks[:, :square_n, :]
        n_tokens = toks.shape[1]
        side = int(math.sqrt(n_tokens))

    pred_depth = F.interpolate(
        toks.mean(dim=-1, keepdim=True).transpose(1, 2).reshape(toks.size(0), 1, side, side),
        size=depth_maps.shape[-2:],
        mode='bilinear',
        align_corners=False,
    ).squeeze(1)

    valid_mask = depth_maps > 0.01
    pred_depth = _normalize_pred_to_gt_range(pred_depth, depth_maps, valid_mask)
    return pred_depth


def _collect_projector_tokens(batch: Dict[str, torch.Tensor], disable_depth: bool = False):
    """Run the actual model forward and capture projector outputs with a forward hook."""
    pixel_values = batch["pixel_values"].to("cuda")
    input_ids = batch["input_ids"].to("cuda")
    attention_mask = batch["attention_mask"].to("cuda")
    depth_maps = batch.get("depth_maps", None)
    if depth_maps is not None:
        depth_maps = depth_maps.to("cuda")

    hook_state = {"output": None, "count": 0}

    def _capture_hook(module, module_inputs, module_output):
        hook_state["output"] = module_output.detach().float()
        hook_state["count"] += 1

    handle = _proj.register_forward_hook(_capture_hook)
    saved_gate = _proj.depth_connector.gate.data.clone()
    try:
        if disable_depth:
            _proj.depth_connector.gate.data.fill_(-100.0)

        with torch.no_grad():
            _ = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                depth_maps=depth_maps,
                use_cache=False,
                return_dict=True,
            )
    finally:
        handle.remove()
        _proj.depth_connector.gate.data.copy_(saved_gate)
        _proj._cached_depth_maps = None

    if hook_state["output"] is None:
        raise RuntimeError("Projector hook did not fire during evaluation forward pass.")

    return hook_state["output"], depth_maps, hook_state["count"]


def _proxy_depth_from_batch(batch: Dict[str, torch.Tensor], disable_depth: bool = False):
    projector_tokens, depth_maps, hook_calls = _collect_projector_tokens(batch, disable_depth=disable_depth)
    if depth_maps is None:
        raise ValueError("depth_maps missing from batch; cannot compute MADD.")
    pred_depth = _tokens_to_proxy_depth(projector_tokens, depth_maps)
    valid_mask = depth_maps > 0.01
    return pred_depth, depth_maps, valid_mask, hook_calls


def _per_sample_madd(pred_depth: torch.Tensor,
                     depth_maps: torch.Tensor,
                     valid_mask: torch.Tensor):
    out = []
    for b in range(pred_depth.size(0)):
        vm = valid_mask[b]
        if vm.sum() == 0:
            continue
        out.append(torch.abs(pred_depth[b][vm].float() - depth_maps[b][vm].float()).mean().item())
    return out


_eval_hook_calls = []
with torch.no_grad():
    for batch in tqdm(eval_loader, desc="Evaluating"):
        if batch.get("depth_maps", None) is None:
            continue

        pred_depth, depth_maps, valid_mask, hook_calls = _proxy_depth_from_batch(batch, disable_depth=False)
        _eval_hook_calls.append(hook_calls)

        madd_metric.update(pred_depth, depth_maps, valid_mask)
        batch_metrics = compute_depth_metrics(pred_depth, depth_maps, valid_mask)
        all_depth_metrics.append(batch_metrics)
        depthon_per_sample_madd.extend(_per_sample_madd(pred_depth, depth_maps, valid_mask))

eval_card = tracker.stop(num_samples=len(val_dataset), tag="evaluation")

# --- Aggregate depth metrics ---
if all_depth_metrics:
    avg_metrics = {}
    for key in all_depth_metrics[0]:
        avg_metrics[key] = np.mean([m[key] for m in all_depth_metrics])

    print(f"\n{'='*60}")
    print(f"  📐 DEPTH EVALUATION RESULTS")
    print(f"{'='*60}")
    print(f"  MADD (primary):     {madd_metric.compute():.4f}")
    print(f"  RMSE:               {avg_metrics.get('rmse', 0):.4f}")
    print(f"  Abs Relative:       {avg_metrics.get('abs_rel', 0):.4f}")
    print(f"  δ < 1.25:           {avg_metrics.get('delta_1.25', 0):.4f}")
    print(f"  δ < 1.25²:          {avg_metrics.get('delta_1.25^2', 0):.4f}")
    print(f"  δ < 1.25³:          {avg_metrics.get('delta_1.25^3', 0):.4f}")
    if _eval_hook_calls:
        print(f"  Projector hook calls/batch: mean={np.mean(_eval_hook_calls):.2f}, min={np.min(_eval_hook_calls)}, max={np.max(_eval_hook_calls)}")
    print(f"{'='*60}")
else:
    avg_metrics = {}

tracker.print_card(eval_card)

if cfg.use_wandb:
    import wandb
    wandb.log({
        "eval/madd": madd_metric.compute(),
        **{f"eval/{k}": v for k, v in avg_metrics.items()},
        "efficiency/evaluation": eval_card,
    })


# --- Optional stress evaluation (never claim-bearing) ---
if stress_dataset is not None:
    print(f"\n{'='*60}")
    print("  🧪 Running STRESS evaluation on NJU2K (toy)")
    print(f"{'='*60}\n")

    stress_loader = DataLoader(stress_dataset, batch_size=cfg.batch_size,
                               collate_fn=collator, shuffle=False)
    stress_madd = MADDMetric()

    with torch.no_grad():
        for batch in tqdm(stress_loader, desc="Stress Evaluating"):
            if batch.get("depth_maps", None) is None:
                continue
            pred_depth, depth_maps, valid_mask, _ = _proxy_depth_from_batch(batch, disable_depth=False)
            stress_madd.update(pred_depth, depth_maps, valid_mask)

    print("\n🧪 Stress MADD:", stress_madd.compute())
else:
    print("\nℹ️ Stress evaluation disabled or NJU2K not available.")


### 8.1b Baseline Evaluation (OBJ-2 / OBJ-3)Runs **zero-shot** (no LoRA) evaluation on the same val set to compute:- MADD delta for OBJ-2 (fine-tuned vs zero-shot)- VRAM/latency delta for OBJ-3 (4-bit vs FP16 estimate)A paired bootstrap significance test is applied to the per-sample MADD values.

In [ ]:
# ============================================================
# P0.5: Baseline evaluation — RGB-only vs Depth-on (OBJ-2/OBJ-3)
# ============================================================
print(f"\n{'='*60}")
print(f"  📐 Running RGB-ONLY vs DEPTH-ON Baseline Evaluation")
print(f"{'='*60}\n")

_proj = _get_projector(model)
_learned_gate = _proj.depth_connector.gate.data.clone()

# ── Baseline A: RGB-only (same projector-output proxy, gate forced near 0) ──
rgbonly_madd_metric = MADDMetric()
rgbonly_depth_metrics = []
rgbonly_per_sample_madd = []
depthon_per_sample_madd = []
_rgb_hook_calls = []
_ft_hook_calls = []

rgbonly_tracker = EfficiencyTracker()
rgbonly_tracker.start()

with torch.no_grad():
    for batch in tqdm(eval_loader, desc="RGB-only baseline"):
        if batch.get("depth_maps", None) is None:
            continue

        # Depth-on: same path as headline MADD
        ft_pred, depth_maps_gpu, valid_mask, ft_hook_calls = _proxy_depth_from_batch(batch, disable_depth=False)
        _ft_hook_calls.append(ft_hook_calls)

        # RGB-only: identical path, but force gate ≈ 0
        bl_pred, _, _, rgb_hook_calls = _proxy_depth_from_batch(batch, disable_depth=True)
        _rgb_hook_calls.append(rgb_hook_calls)

        rgbonly_per_sample_madd.extend(_per_sample_madd(bl_pred, depth_maps_gpu, valid_mask))
        depthon_per_sample_madd.extend(_per_sample_madd(ft_pred, depth_maps_gpu, valid_mask))

        rgbonly_madd_metric.update(bl_pred, depth_maps_gpu, valid_mask)
        rgbonly_depth_metrics.append(compute_depth_metrics(bl_pred, depth_maps_gpu, valid_mask))

# Restore gate
_proj.depth_connector.gate.data.copy_(_learned_gate)

rgbonly_card = rgbonly_tracker.stop(num_samples=len(val_dataset), tag="rgb-only-baseline")

# ── Aggregate ──
bl_avg = {}
if rgbonly_depth_metrics:
    for key in rgbonly_depth_metrics[0]:
        bl_avg[key] = np.mean([m[key] for m in rgbonly_depth_metrics])

bl_madd = rgbonly_madd_metric.compute()
ft_madd = madd_metric.compute()

print(f"\n{'='*60}")
print(f"  📐 RGB-ONLY vs DEPTH-ON COMPARISON")
print(f"{'='*60}")
print(f"  {'Metric':<20s}  {'RGB-only':>12s}  {'Depth-on':>12s}  {'Δ':>10s}")
print(f"  {'─'*56}")
print(f"  {'MADD (↓)':<20s}  {bl_madd:>12.4f}  {ft_madd:>12.4f}  {ft_madd - bl_madd:>+10.4f}")
if avg_metrics and bl_avg:
    for k in ["rmse", "abs_rel", "delta_1.25"]:
        bv = bl_avg.get(k, 0)
        fv = avg_metrics.get(k, 0)
        arrow = "↓" if k != "delta_1.25" else "↑"
        print(f"  {k+' ('+arrow+')':<20s}  {bv:>12.4f}  {fv:>12.4f}  {fv - bv:>+10.4f}")
if _rgb_hook_calls and _ft_hook_calls:
    print(f"  {'Hook calls/batch':<20s}  {np.mean(_rgb_hook_calls):>12.2f}  {np.mean(_ft_hook_calls):>12.2f}  {np.mean(_ft_hook_calls) - np.mean(_rgb_hook_calls):>+10.2f}")
print(f"{'='*60}")

# ── OBJ-3: VRAM comparison (4-bit vs FP16) ──
# FP16 baseline is NOT available in this run — mark as BLOCKED
_fp16_baseline_available = False
print(f"\n  ⚡ EFFICIENCY COMPARISON (OBJ-3)")
print(f"  {'':20s}  {'RGB-only':>12s}  {'Depth-on':>12s}  {'Δ%':>10s}")
print(f"  {'─'*56}")
bl_vram = rgbonly_card["peak_vram_mb"]
ft_vram = eval_card["peak_vram_mb"]
delta_pct = ((bl_vram - ft_vram) / max(bl_vram, 1)) * 100
print(f"  {'Peak VRAM (MB)':<20s}  {bl_vram:>12.0f}  {ft_vram:>12.0f}  {delta_pct:>+9.1f}%")
bl_lat = rgbonly_card["latency_ms"]
ft_lat = eval_card["latency_ms"]
lat_delta = ((bl_lat - ft_lat) / max(bl_lat, 1)) * 100
print(f"  {'Latency (ms/samp)':<20s}  {bl_lat:>12.1f}  {ft_lat:>12.1f}  {lat_delta:>+9.1f}%")

obj3_geo_pass = abs(ft_madd - bl_madd) / max(bl_madd, 1e-6) <= 0.05
print(f"\n  OBJ-3 VRAM ≥15% vs FP16:  ⛔ BLOCKED (FP16 baseline not run)")
print(f"  OBJ-3 geo consistency ≤5%: {'✅ PASS' if obj3_geo_pass else '❌ FAIL'} ({abs(ft_madd-bl_madd)/max(bl_madd,1e-6)*100:.1f}%)")

# ── Paired significance tests (OBJ-2) using the SAME normalized per-image proxy MADD ──
from scipy import stats as scipy_stats
baseline_arr = np.array(rgbonly_per_sample_madd, dtype=np.float64)
finetuned_arr = np.array(depthon_per_sample_madd, dtype=np.float64)
n_paired = min(len(baseline_arr), len(finetuned_arr))

if n_paired >= 4:
    baseline_arr = baseline_arr[:n_paired]
    finetuned_arr = finetuned_arr[:n_paired]
    diffs = baseline_arr - finetuned_arr  # positive = depth-on better

    n_bootstrap = 10000
    rng = np.random.RandomState(cfg.seed)
    boot_means = np.array([
        rng.choice(diffs, size=n_paired, replace=True).mean()
        for _ in range(n_bootstrap)
    ])
    p_value_boot = (boot_means <= 0).mean()

    if n_paired >= 6:
        try:
            stat_w, p_value_wilcoxon = scipy_stats.wilcoxon(
                baseline_arr,
                finetuned_arr,
                alternative="greater"
            )
        except Exception:
            p_value_wilcoxon = float("nan")
    else:
        p_value_wilcoxon = float("nan")

    print(f"\n  📊 STATISTICAL TEST (OBJ-2): depth-on < rgb-only MADD?")
    print(f"     Paired samples:      {n_paired}")
    print(f"     Mean MADD diff:      {diffs.mean():.4f} (positive = depth-on better)")
    print(f"     Diff min / max:      {diffs.min():.4f} / {diffs.max():.4f}")
    print(f"     All diffs <= 0:      {bool(np.all(diffs <= 0))}")
    print(f"     RGB-only mean:       {baseline_arr.mean():.4f}")
    print(f"     Depth-on mean:       {finetuned_arr.mean():.4f}")
    print(f"     Bootstrap p-value:   {p_value_boot:.4f}")
    print(f"     Wilcoxon p-value:    {p_value_wilcoxon:.4f}")
    sig = p_value_boot < 0.05
    print(f"     Significant (p<0.05): {'✅ YES' if sig else '❌ NO'}")
    if SMOKE_RUN:
        print(f"     ⚠️  SMOKE RUN: n={n_paired} is too small for meaningful significance.")
else:
    p_value_boot = float("nan")
    p_value_wilcoxon = float("nan")
    sig = False
    print("  ⚠️  Too few paired samples for significance test.")

# Store for claim-evidence table
_baseline_results = {
    "baseline_type": "rgb_only_gate_forced_same_proxy",
    "rgb_only_madd": bl_madd,
    "depthon_madd": ft_madd,
    "zero_shot_madd": bl_madd,       # backward compat key
    "finetuned_madd": ft_madd,       # backward compat key
    "madd_delta": ft_madd - bl_madd,
    "p_value_bootstrap": float(p_value_boot),
    "p_value_wilcoxon": float(p_value_wilcoxon),
    "significant": bool(sig),
    "n_paired": n_paired,
    "vram_reduction_pct": delta_pct,
    "fp16_baseline_available": _fp16_baseline_available,
    "geo_consistency_pct": abs(ft_madd - bl_madd) / max(bl_madd, 1e-6) * 100,
    "baseline_card": rgbonly_card,
}

gc.collect()
torch.cuda.empty_cache()
print("\n✅ Baseline comparison complete")


### 8.1c SUN RGB-D Alignment Benchmark (OBJ-2)Evaluates depth-aware instruction alignment on SUN RGB-D test split.Required by OBJ-2: statistically significant improvement vs zero-shot on SUN RGB-D.

In [ ]:
# ============================================================
# P0.6: SUN RGB-D Alignment Accuracy Benchmark (OBJ-2)
# Uses scene labels as deterministic supervised target.
# ============================================================
print(f"\n{'='*60}")
print(f"  🏛️  SUN RGB-D ALIGNMENT ACCURACY BENCHMARK (OBJ-2)")
print(f"{'='*60}\n")

_sun_eval_results = None
try:
    sun_eval_dataset = SUNRGBDInstructionDataset(
        processor=processor,
        split="test",
        max_samples=cfg.sunrgbd_max_samples,
        image_size=cfg.image_size,
        depth_size=cfg.depth_size,
    )
    sun_eval_loader = DataLoader(
        sun_eval_dataset, batch_size=cfg.batch_size,
        collate_fn=collator, shuffle=False
    )

    model.eval()
    _proj = _get_projector(model)
    _learned_gate = _proj.depth_connector.gate.data.clone()

    from PIL import Image as _PILImage
    _dummy_img = _PILImage.new("RGB", (cfg.image_size, cfg.image_size))

    def _generate_scene_label(batch, disable_depth=False):
        """Generate scene label predictions for a batch."""
        labels = []
        pixel_values = batch["pixel_values"].to("cuda")
        depth_maps = batch.get("depth_maps", None)

        if disable_depth:
            _proj.depth_connector.gate.data.fill_(-100.0)
        else:
            _proj.depth_connector.gate.data.copy_(_learned_gate)

        for b_idx in range(pixel_values.size(0)):
            prompt = "USER: <image>\nWhat type of room or scene is this? Answer with one label.\nASSISTANT:"
            inputs = processor(text=prompt, images=_dummy_img, return_tensors="pt",
                               padding=True, truncation=True, max_length=cfg.max_length)
            input_ids = inputs["input_ids"].to("cuda")
            attention_mask = inputs["attention_mask"].to("cuda")

            with torch.no_grad():
                # Inject depth if available
                if depth_maps is not None and not disable_depth:
                    _proj._cached_depth_maps = depth_maps[b_idx:b_idx+1].to("cuda")

                out = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    pixel_values=pixel_values[b_idx:b_idx+1],
                    max_new_tokens=32,
                    do_sample=False,
                )

            new_tokens = out[0, input_ids.shape[-1]:]
            decoded = processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
            # Extract first meaningful word/phrase
            decoded = decoded.split(".")[0].split(",")[0].strip()
            labels.append(decoded)

        # Restore gate
        _proj.depth_connector.gate.data.copy_(_learned_gate)
        return labels

    # ── Extract ground-truth scene labels ──
    gt_labels = []
    ft_preds = []
    zs_preds = []

    for batch in tqdm(sun_eval_loader, desc="SUN RGB-D accuracy eval"):
        n_batch = batch["pixel_values"].size(0)

        # Ground truth: extract from the Q/A template in the dataset
        # The scene label is embedded in the answer text; we extract it.
        # For simplicity in SMOKE, we use the first template's scene keyword.
        for b_idx in range(n_batch):
            # Try to get scene label from dataset metadata
            # Fall back to a generic label extraction from labels tensor
            gt_labels.append("scene")  # placeholder — refined below

        # Fine-tuned predictions (depth-on)
        ft_preds.extend(_generate_scene_label(batch, disable_depth=False))

        # Zero-shot predictions (depth-off / LoRA disabled not practical here,
        # so we use RGB-only as zero-shot proxy)
        zs_preds.extend(_generate_scene_label(batch, disable_depth=True))

    # ── Attempt to extract real GT labels from dataset ──
    real_gt = []
    for i in range(len(sun_eval_dataset)):
        try:
            sample = sun_eval_dataset[i]
            if hasattr(sun_eval_dataset, 'scene_labels') and i < len(sun_eval_dataset.scene_labels):
                real_gt.append(sun_eval_dataset.scene_labels[i].lower().strip())
            elif hasattr(sample, 'get') and 'scene_label' in sample:
                real_gt.append(sample['scene_label'].lower().strip())
            else:
                real_gt.append("unknown")
        except Exception:
            real_gt.append("unknown")

    if len(real_gt) == len(ft_preds):
        gt_labels = real_gt

    # ── Compute accuracy (exact match after normalization) ──
    def _normalize_label(s):
        return s.lower().strip().replace("_", " ").replace("-", " ")

    n_eval = min(len(gt_labels), len(ft_preds), len(zs_preds))

    ft_correct = [int(_normalize_label(gt_labels[i]) in _normalize_label(ft_preds[i])
                      or _normalize_label(ft_preds[i]) in _normalize_label(gt_labels[i]))
                  for i in range(n_eval)]
    zs_correct = [int(_normalize_label(gt_labels[i]) in _normalize_label(zs_preds[i])
                      or _normalize_label(zs_preds[i]) in _normalize_label(gt_labels[i]))
                  for i in range(n_eval)]

    acc_finetuned = sum(ft_correct) / max(n_eval, 1)
    acc_zeroshot = sum(zs_correct) / max(n_eval, 1)

    # ── McNemar test (paired binary correctness) ──
    sun_p_mcnemar = float("nan")
    sun_sig = False
    if n_eval >= 4:
        # Contingency: ft_correct vs zs_correct
        b_val = sum(1 for i in range(n_eval) if ft_correct[i] == 1 and zs_correct[i] == 0)
        c_val = sum(1 for i in range(n_eval) if ft_correct[i] == 0 and zs_correct[i] == 1)
        if b_val + c_val > 0:
            # McNemar chi-squared (with continuity correction)
            chi2 = (abs(b_val - c_val) - 1) ** 2 / max(b_val + c_val, 1)
            sun_p_mcnemar = 1.0 - scipy_stats.chi2.cdf(chi2, df=1)
            sun_sig = sun_p_mcnemar < 0.05

    print(f"\n  SUN RGB-D Alignment Accuracy Results:")
    print(f"  {'Metric':<25s}  {'RGB-only':>12s}  {'Depth-on':>12s}")
    print(f"  {'─'*51}")
    print(f"  {'Accuracy':<25s}  {acc_zeroshot:>12.4f}  {acc_finetuned:>12.4f}")
    print(f"\n  Eval samples:      {n_eval}")
    print(f"  McNemar p-value:   {sun_p_mcnemar:.4f}")
    print(f"  Significant:       {'✅ YES' if sun_sig else '❌ NO'}")
    if SMOKE_RUN:
        print(f"  ⚠️  SMOKE RUN: toy sample size — not claim-bearing.")

    _sun_eval_results = {
        "dataset": "SUN RGB-D",
        "metric": "alignment_accuracy",
        "n_samples": n_eval,
        "acc_finetuned": acc_finetuned,
        "acc_zeroshot": acc_zeroshot,
        "p_value_mcnemar": float(sun_p_mcnemar),
        "significant": bool(sun_sig),
    }

except Exception as e:
    print(f"  ⚠️  SUN RGB-D evaluation failed: {e}")
    import traceback; traceback.print_exc()
    _sun_eval_results = {"status": "failed", "error": str(e)[:200]}

print("\n✅ SUN RGB-D benchmark complete")


### 8.2 Qualitative Generation Test
Tests the fine-tuned model on depth-aware questions to validate instruction following.

In [ ]:
test_questions = [
    "Describe the salient objects in this image.",
    "What regions of this image stand out the most?",
    "Identify the most visually prominent areas in this scene.",
]

In [ ]:
# ============================================================
# Qualitative test: generate responses to depth questions
# ============================================================
print(f"\n{'='*60}")
print(f"  🗣️  Qualitative Generation Test")
print(f"{'='*60}\n")

from PIL import Image

model.eval()
_proj = _get_projector(model)

test_sample = val_dataset[0]

# Ensure pixel_values is exactly 4D: (1, 3, H, W)
pv = test_sample["pixel_values"]
if pv.dim() == 4:        # (1, 3, H, W) from processor
    pixel_values = pv.to("cuda")
elif pv.dim() == 3:       # (3, H, W)
    pixel_values = pv.unsqueeze(0).to("cuda")
else:
    raise ValueError(f"Unexpected pixel_values shape: {pv.shape}")

# Depth maps (if available)
depth_maps = test_sample.get("depth_maps", None)
if depth_maps is not None:
    if depth_maps.dim() == 2:      # (H, W)
        depth_maps = depth_maps.unsqueeze(0).to("cuda")   # (1, H, W)
    elif depth_maps.dim() == 3:    # (1, H, W)
        depth_maps = depth_maps.to("cuda")
    else:
        depth_maps = depth_maps.to("cuda")

dummy_image = Image.new("RGB", (cfg.image_size, cfg.image_size))

for q in test_questions[:2 if SMOKE_RUN else 3]:
    prompt = f"USER: <image>\n{q}\nASSISTANT:"
    inputs = processor(text=prompt, images=dummy_image, return_tensors="pt",
                       padding=True, truncation=True, max_length=cfg.max_length)
    input_ids = inputs["input_ids"].to("cuda")
    attention_mask = inputs["attention_mask"].to("cuda")

    with torch.no_grad():
        # Inject depth into projector cache before generate
        if depth_maps is not None and _proj is not None:
            _proj._cached_depth_maps = depth_maps

        out = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            max_new_tokens=128,
            do_sample=False,
        )

    new_tokens = out[0, input_ids.shape[-1]:]
    decoded = processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    print(f"\nQ: {q}")
    print(f"A: {decoded[:400] if decoded else '[empty — expected on smoke run with minimal training]'}")

In [ ]:
# # Use a dummy image so processor expands <image> → 576 patch tokens

# dummy_image = Image.new("RGB", (cfg.image_size, cfg.image_size))

# for q in test_questions[:2 if SMOKE_RUN else 3]:
#      prompt = f"USER: <image>\n{q}\nASSISTANT:"
#      inputs = processor(text=prompt, images=dummy_image, return_tensors="pt",
#                         padding=True, truncation=True, max_length=cfg.max_length)
#      input_ids = inputs["input_ids"].to("cuda")
#      attention_mask = inputs["attention_mask"].to("cuda")

#      with torch.no_grad():
#          out = model.generate(
#              input_ids=input_ids,
#              attention_mask=attention_mask,
#              pixel_values=pixel_values,   # real values from test_sample
#              max_new_tokens=128,
#              do_sample=False,
#          )

## 9. Visualization & Analysis

In [ ]:
# ============================================================
# Depth map visualization + training curves
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: Sample RGB + Depth + Depth histogram
for i, idx in enumerate(range(min(3, len(val_dataset)))):
    sample = val_dataset[idx]
    depth = sample["depth_map"].numpy()

    if i == 0:
        ax = axes[0, 0]
        ax.set_title("Depth Map (sample 0)", fontsize=11)
        im = ax.imshow(depth, cmap="plasma")
        plt.colorbar(im, ax=ax, fraction=0.046)
        ax.axis("off")

    if i == 0:
        ax = axes[0, 1]
        ax.set_title("Depth Histogram", fontsize=11)
        valid_depth = depth[depth > 0.01]
        ax.hist(valid_depth.flatten(), bins=50, color="steelblue", alpha=0.7)
        ax.axvline(np.mean(valid_depth), color="red", linestyle="--",
                   label=f"Mean: {np.mean(valid_depth):.2f}")
        ax.set_xlabel("Depth (normalized)")
        ax.set_ylabel("Count")
        ax.legend()

# Efficiency comparison
ax = axes[0, 2]
ax.set_title("VRAM Usage", fontsize=11)
cards = tracker.summary()
if cards:
    tags = [c["tag"] for c in cards]
    vrams = [c["peak_vram_mb"] for c in cards]
    colors = ["#2196F3", "#4CAF50", "#FF9800"][:len(tags)]
    ax.barh(tags, vrams, color=colors)
    ax.set_xlabel("Peak VRAM (MB)")
    for i, v in enumerate(vrams):
        ax.text(v + 50, i, f"{v:.0f} MB", va="center", fontsize=9)

# Row 2: Training loss (if available)
ax = axes[1, 0]
ax.set_title("Training Loss", fontsize=11)
if hasattr(trainer.state, 'log_history') and trainer.state.log_history:
    losses = [(h.get("step", 0), h["loss"])
              for h in trainer.state.log_history if "loss" in h]
    if losses:
        steps, loss_vals = zip(*losses)
        ax.plot(steps, loss_vals, "b-o", markersize=3)
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
    else:
        ax.text(0.5, 0.5, "No loss data", ha="center", va="center",
                transform=ax.transAxes)
else:
    ax.text(0.5, 0.5, "No training history", ha="center", va="center",
            transform=ax.transAxes)

# MADD results bar
ax = axes[1, 1]
ax.set_title("Depth Metrics", fontsize=11)
if all_depth_metrics:
    metric_names = ["madd", "rmse", "abs_rel"]
    metric_vals = [avg_metrics[k] for k in metric_names]
    ax.bar(metric_names, metric_vals, color=["#E91E63", "#9C27B0", "#FF5722"])
    ax.set_ylabel("Value (lower is better)")
    for i, v in enumerate(metric_vals):
        ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)

# Delta thresholds
ax = axes[1, 2]
ax.set_title("δ Accuracy Thresholds", fontsize=11)
if all_depth_metrics:
    delta_names = ["δ<1.25", "δ<1.25²", "δ<1.25³"]
    delta_vals = [avg_metrics.get(f"delta_1.25{s}", 0) for s in ["", "^2", "^3"]]
    ax.bar(delta_names, delta_vals, color=["#00BCD4", "#009688", "#4CAF50"])
    ax.set_ylim(0, 1)
    ax.set_ylabel("Accuracy (higher is better)")
    for i, v in enumerate(delta_vals):
        ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(cfg.output_dir, "evaluation_dashboard.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Dashboard saved to {cfg.output_dir}/evaluation_dashboard.png")


## 10. Claim-to-Evidence Table (WP3 Deliverable)
Maps every research claim to empirical evidence. Required for final thesis submission.

In [ ]:
# ============================================================
# WP3: Claim-to-Evidence Table (P0.1 + P0.7: W&B traceability + gating)
# ============================================================

# Determine actual dataset used
_train_ds_name = cfg.train_dataset_name
_eval_ds_name = cfg.eval_dataset_name
_is_toy = _train_ds_name != "nyuv2"
_fp16_avail = _baseline_results.get("fp16_baseline_available", False)

# ── P0.1: structured W&B provenance ──
if cfg.use_wandb and cfg.wandb_run_id:
    _wandb_info = {
        "id": cfg.wandb_run_id,
        "name": cfg.wandb_run_name,
        "url": cfg.wandb_run_url,
        "project": cfg.wandb_project_name,
        "entity": cfg.wandb_entity,
    }
else:
    _wandb_info = {"status": "disabled"}

# ── P0.7: gating helper ──
def _gate_status(raw_status, claim_id):
    """Enforce evidence gates: toy/smoke data → NON-CLAIM-BEARING."""
    if SMOKE_RUN or _is_toy:
        return f"NON-CLAIM-BEARING (toy/smoke) [{raw_status}]"
    if claim_id in ("C1",) and not _fp16_avail:
        return f"BLOCKED (FP16 baseline missing) [{raw_status}]"
    return raw_status

claim_evidence = {
    "metadata": {
        "version": "v0.5-P0",
        "smoke_run": SMOKE_RUN,
        "train_dataset": _train_ds_name,
        "eval_dataset": _eval_ds_name,
        "is_toy_data": _is_toy,
        "fp16_baseline_available": _fp16_avail,
        "wandb": _wandb_info,
        "note": "Claims on toy data are NOT thesis-bearing. Re-run with SMOKE_RUN=False for valid evidence."
                if (_is_toy or SMOKE_RUN) else "Evidence collected on primary dataset.",
    },
    "claims": [
        {
            "id": "C1",
            "claim": "QLoRA 4-bit reduces VRAM usage by ≥15% vs FP16 baseline",
            "objective": "OBJ-3",
            "metric": "peak_vram_mb",
            "dataset": _eval_ds_name,
            "is_toy_data": _is_toy,
            "smoke_run": SMOKE_RUN,
            "baseline_available": True,
            "fp16_available": _fp16_avail,
            "evidence": (
                f"BLOCKED — FP16 baseline not executed in this run. "
                f"4-bit VRAM: {eval_card['peak_vram_mb']:.0f} MB, "
                f"RGB-only baseline VRAM: {_baseline_results['baseline_card']['peak_vram_mb']:.0f} MB"
            ) if not _fp16_avail else (
                f"4-bit VRAM: {eval_card['peak_vram_mb']:.0f} MB, "
                f"FP16 VRAM: TBD MB, Reduction: TBD%"
            ),
            "status": _gate_status(
                "BLOCKED (FP16 baseline missing)" if not _fp16_avail else "PENDING",
                "C1"
            ),
            "wandb": _wandb_info,
        },
        {
            "id": "C2",
            "claim": "Depth-on achieves significant improvement over RGB-only baseline",
            "objective": "OBJ-2",
            "metric": "madd",
            "dataset": _eval_ds_name,
            "is_toy_data": _is_toy,
            "smoke_run": SMOKE_RUN,
            "baseline_available": True,
            "fp16_available": _fp16_avail,
            "evidence": (
                f"Depth-on MADD: {_baseline_results['depthon_madd']:.4f}, "
                f"RGB-only MADD: {_baseline_results['rgb_only_madd']:.4f}, "
                f"p(bootstrap): {_baseline_results['p_value_bootstrap']:.4f}, "
                f"n={_baseline_results['n_paired']}"
            ),
            "status": _gate_status(
                "✅ SIGNIFICANT" if _baseline_results["significant"]
                else "⚠️ NOT SIGNIFICANT",
                "C2"
            ),
            "wandb": _wandb_info,
        },
        {
            "id": "C2b",
            "claim": "Depth-on achieves significant alignment accuracy improvement on SUN RGB-D",
            "objective": "OBJ-2",
            "metric": "alignment_accuracy",
            "dataset": "SUN RGB-D",
            "is_toy_data": _is_toy,
            "smoke_run": SMOKE_RUN,
            "baseline_available": True,
            "fp16_available": _fp16_avail,
            "evidence": (
                f"acc_finetuned: {_sun_eval_results.get('acc_finetuned', 'N/A')}, "
                f"acc_zeroshot: {_sun_eval_results.get('acc_zeroshot', 'N/A')}, "
                f"p(McNemar): {_sun_eval_results.get('p_value_mcnemar', 'N/A')}"
                if _sun_eval_results and _sun_eval_results.get("metric") == "alignment_accuracy"
                else f"Results: {json.dumps(_sun_eval_results, default=str)}"
            ),
            "status": _gate_status(
                "✅ SIGNIFICANT" if (_sun_eval_results or {}).get("significant", False)
                else "⚠️ NOT SIGNIFICANT",
                "C2b"
            ),
            "wandb": _wandb_info,
        },
        {
            "id": "C3",
            "claim": "Pipeline achieves ≥85% stability in data ingestion",
            "objective": "OBJ-1",
            "metric": "pipeline_stability_rate",
            "dataset": _train_ds_name,
            "is_toy_data": _is_toy,
            "smoke_run": SMOKE_RUN,
            "baseline_available": True,
            "fp16_available": _fp16_avail,
            "evidence": (
                f"Stability: {stability_tracker.stability_rate()*100:.1f}% "
                f"({json.dumps({k: v['stability_rate'] for k,v in stability_tracker.summary().items() if k != 'overall_stability_rate'})})"
            ),
            "status": _gate_status(
                "✅ PASS" if stability_tracker.meets_obj1() else "❌ FAIL",
                "C3"
            ),
            "note": f"OBJ-1 requires NYUv2. Current dataset: {_train_ds_name}." if _is_toy else None,
            "wandb": _wandb_info,
        },
        {
            "id": "C4",
            "claim": "Late fusion (depth adapter) preserves geometric consistency within 5%",
            "objective": "OBJ-3",
            "metric": "madd_delta_pct",
            "dataset": _eval_ds_name,
            "is_toy_data": _is_toy,
            "smoke_run": SMOKE_RUN,
            "baseline_available": True,
            "fp16_available": _fp16_avail,
            "evidence": (
                f"Geo consistency: {_baseline_results['geo_consistency_pct']:.1f}% "
                f"(MADD delta: {_baseline_results['madd_delta']:.4f})"
            ),
            "status": _gate_status(
                "✅ PASS" if _baseline_results["geo_consistency_pct"] <= 5
                else "❌ FAIL",
                "C4"
            ),
            "wandb": _wandb_info,
        },
    ]
}

# Remove None notes
for c in claim_evidence["claims"]:
    if "note" in c and c["note"] is None:
        del c["note"]

print(f"\n{'='*70}")
print(f"  📋 CLAIM-TO-EVIDENCE TABLE (v0.5-P0)")
print(f"  Dataset provenance: train={_train_ds_name}, eval={_eval_ds_name}")
print(f"  W&B: {_wandb_info.get('url', 'disabled')}")
print(f"{'='*70}")
for c in claim_evidence["claims"]:
    print(f"\n  [{c['id']}] {c['claim']}")
    print(f"       Objective: {c['objective']} | Metric: {c['metric']} | Dataset: {c.get('dataset','—')}")
    print(f"       Evidence:  {c['evidence'][:120]}")
    print(f"       Status:    {c['status']}")
    if "note" in c:
        print(f"       Note:      {c['note']}")
print(f"\n{'='*70}")

# Save as JSON
import json
with open(os.path.join(cfg.output_dir, "claim_evidence_table.json"), "w") as f:
    json.dump(claim_evidence, f, indent=2, default=str)

# Also save stability report
with open(os.path.join(cfg.output_dir, "stability_report.json"), "w") as f:
    json.dump(stability_tracker.summary(), f, indent=2)

# Also save baseline comparison
with open(os.path.join(cfg.output_dir, "baseline_comparison.json"), "w") as f:
    json.dump(_baseline_results, f, indent=2, default=str)

if _sun_eval_results:
    with open(os.path.join(cfg.output_dir, "sun_rgbd_results.json"), "w") as f:
        json.dump(_sun_eval_results, f, indent=2, default=str)

print(f"\n✅ Evidence artifacts saved to {cfg.output_dir}/")


## 11. Run Summary & Next Steps

In [ ]:
# ============================================================
# Final Summary (v0.5-P0 — depth fusion wired, evidence gated)
# ============================================================

_train_ds = cfg.train_dataset_name
_eval_ds = cfg.eval_dataset_name
_alpha_final = torch.sigmoid(_get_projector(model).depth_connector.gate).item()

print(f"""
{'='*60}
  🏁 RUN COMPLETE — {'SMOKE RUN' if SMOKE_RUN else 'FULL TRAINING'} (v0.5-P0)
{'='*60}

  📦 Model:      {cfg.base_model}
  🔧 Method:     QLoRA (4-bit NF4) + LoRA r={cfg.lora_r} + DepthAwareProjector
  📊 Train data: {_train_ds} ({len(train_subset)} samples)
  📊 Eval data:  {_eval_ds} ({len(val_dataset)} samples)
  🏷️  Provenance: {'⚠️ TOY DATA — not claim-bearing' if _train_ds != 'nyuv2' else '✅ Primary dataset (NYUv2)'}
  🔗 W&B:        {cfg.wandb_run_url or 'disabled'}

  ── Training ──────────────────────────────
  Loss (final):  {metrics.get('train_loss', 'N/A')}
  Time:          {train_card['total_time_s']:.1f}s
  Peak VRAM:     {train_card['peak_vram_mb']:.0f} MB
  Depth gate α:  {_alpha_final:.6f}

  ── Evaluation ────────────────────────────
  MADD (depth-on):  {madd_metric.compute():.4f}
  MADD (rgb-only):  {_baseline_results['rgb_only_madd']:.4f}
  Δ MADD:           {_baseline_results['madd_delta']:+.4f}
  p-value:          {_baseline_results['p_value_bootstrap']:.4f}
  Eval VRAM:        {eval_card['peak_vram_mb']:.0f} MB

  ── OBJ Status ────────────────────────────
  OBJ-1 (stability ≥85%): {'✅' if stability_tracker.meets_obj1() else '❌'} {stability_tracker.stability_rate()*100:.1f}%{' (on '+_train_ds+', need NYUv2)' if _train_ds != 'nyuv2' else ''}
  OBJ-2 (p<0.05 depth>rgb): {'✅' if _baseline_results['significant'] else '❌'} p={_baseline_results['p_value_bootstrap']:.4f}
  OBJ-3 (VRAM ≥15% vs FP16): ⛔ BLOCKED (FP16 baseline not run)
  OBJ-3 (geo ≤5%):           {'✅' if _baseline_results['geo_consistency_pct'] <= 5 else '❌'} {_baseline_results['geo_consistency_pct']:.1f}%

  ── Outputs ───────────────────────────────
  Adapters:          {cfg.output_dir}/lora_adapters/
  Depth modules:     {cfg.output_dir}/depth_modules.pt
  Dashboard:         {cfg.output_dir}/evaluation_dashboard.png
  Evidence table:    {cfg.output_dir}/claim_evidence_table.json
  W&B provenance:    {cfg.output_dir}/wandb_run.json
  Stability report:  {cfg.output_dir}/stability_report.json
  Baseline compare:  {cfg.output_dir}/baseline_comparison.json

{'='*60}
""")

if SMOKE_RUN:
    print("""
  ⚡ SMOKE RUN completed successfully!
     Pipeline validated end-to-end. To run full training:

     1. Set SMOKE_RUN = False in Cell 0
     2. Re-run all cells
     3. Expected full run time: ~2-4 hours on T4

  📋 P0 PATCHES APPLIED (v0.5-P0):
     ✅ P0.1: W&B run.id + URL traceability in claim table
     ✅ P0.2: Depth fusion wired into LLaVA forward/backprop
     ✅ P0.3: Depth modules trainable + gate logging
     ✅ P0.4: Stability probe covers ingest/tokenize/forward/backward
     ✅ P0.5: RGB-only baseline replaces random encoder; FP16 claim gated
     ✅ P0.6: SUN RGB-D uses alignment accuracy + McNemar test
     ✅ P0.7: Claim table gates: toy/smoke → NON-CLAIM-BEARING

  📋 REMAINING FOR FULL RUN:
     ☐ Run with SMOKE_RUN=False on NYUv2 (OBJ-1 requires NYUv2)
     ☐ Collect FP16 baseline (separate run without quantization)
     ☐ Achieve p<0.05 on SUN RGB-D with full data (OBJ-2)
""")

# Cleanup
if cfg.use_wandb:
    import wandb
    wandb.finish()
    print("✅ W&B run finalized")

gc.collect()
torch.cuda.empty_cache()
print("✅ GPU memory cleared")


In [ ]:
import shutil
from google.colab import files
# Zip the results folder
shutil.make_archive("mm-sod-output", 'zip', "mm-sod-output")
# Download the zip file
files.download("mm-sod-output.zip")
print(f"\nDownloaded the output folder!")